# ⛓️ 20 · Graphical models: Bayesian networks and Markov chains
### Bayesian networks: from factorisations to graphs and back · conditional probability tables · inference by enumeration (the bronchitis network) · directed, undirected and factor graphs · the Markov blanket · Markov chains: transition matrices, CpG islands, n-step probabilities and the long run

**BiologyToMath** · Part E — Probability · Module 20 of 28

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/math/tutorial/BiologyToMath/notebooks/20_bayesian_networks_markov_chains.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib (the course's `draw_graph` for networks), SciPy/NumPy linear algebra for eigenvectors |
| **Prerequisites** | [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb) (joint, marginal and conditional distributions; independence); [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) (chain rule, conditional independence); [Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb) (directed graphs, in-neighbours); [Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb) (matrix products) and [Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb) (eigenvectors) for Sections 9–10 |
| **Estimated time** | 170 min (reading + running + the ✏️ exercises) |
| **02-680 topics covered** | Graphical representations of random variables. **Bayesian networks** (directed graphical models): why graphs (compact specification, visual dependencies, model design, conditional independence by inspection, computation as graph manipulation); arrows as conditional probabilities and when they may be read causally; constructing the network from a factorised joint distribution, how the graph depends on the factorisation, and reading the joint distribution off a graph (the five-node example, p(x) = ∏ p(x_k | Pa_k) with parents = in-neighbours); conditional probability tables (X → Y → Z with and without X → Z); marginal independence of two parents (B → A ← E); the bronchitis network and P(T | S) = 0.30; directed graphical models, Markov random fields and factor graphs; the **Markov blanket**. **Markov chains**: stochastic processes, the Markov property, the chain-rule factorisation, finite chains and transition matrices; CpG islands versus background DNA (the sequence ACTTC); the telephone-lines and weather examples; n-step transition probabilities and stationary distributions; applications in genomics (motifs, hidden Markov models for genes, recombination and haplotypes) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. The conditional probability tables of the X/Y/Z networks, the burglary–earthquake–alarm network, the bronchitis network and the Markov-blanket network are **illustrative teaching numbers**, not clinical or epidemiological estimates. The CpG-island and background transition matrices are the classic rounded values of Durbin *et al.* (1998), **typed in** (they live in the course toolkit); the telephone-line and weather chains are classic textbook examples, **typed in**. All DNA sequences are **SIMULATED** with `simulate_dna`, and every sampled data set is **SIMULATED** from the stated model.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Construct** the Bayesian network of a factorised joint distribution and **write** the factorisation
   $p(\mathbf x) = \prod_k p(x_k \mid \mathrm{Pa}_k)$ of a given directed acyclic graph.
2. **Compute** a full joint distribution, marginals and conditionals from conditional probability tables, and
   **count** the parameters a network saves compared with a full joint table.
3. **Derive** step by step, naming the rule used in each line, probabilities such as $P(T \mid S) = 0.30$ in the
   bronchitis network, and **answer** further queries by enumeration and by simulation.
4. **Prove** that two parents with no common ancestor are marginally independent, and **demonstrate** that they become
   dependent once their common child is observed.
5. **Distinguish** directed graphical models, Markov random fields and factor graphs, and **find** the Markov blanket of
   a node — then **verify** numerically that it shields the node from the rest of the network.
6. **Define** a Markov chain, **derive** its joint distribution from the chain rule plus the Markov assumption, and
   **use** two transition matrices to decide whether a DNA sequence looks like a CpG island.
7. **Compute** n-step transition probabilities with matrix powers and the stationary distribution as a left
   eigenvector, and **interpret** them for the telephone-lines, weather and DNA examples.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, course helpers, a simulated genome with a hidden CpG island | 5 min |
| 1 · Why graphs? | the parameter explosion; arrows as conditional probabilities | 10 min |
| 2 · Factorisation ⇄ graph | build networks from factorisations and read factorisations off graphs | 20 min |
| 3 · Conditional probability tables | the X → Y → Z networks: joints, marginals, parameters, sampling | 20 min |
| 4 · Two causes, one effect | marginal independence of parents; explaining away | 10 min |
| 5 · The bronchitis network | $P(T \mid S)$ step by step; inference by enumeration | 20 min |
| 6 · Three kinds of graphical model | directed, undirected (Markov random fields), factor graphs | 10 min |
| 7 · The Markov blanket | parents, children, co-parents — verified numerically | 15 min |
| 8 · Markov chains | the Markov property, factorisation, transition matrices | 15 min |
| 9 · CpG islands | ACTTC, log-odds scores, scanning a genome | 20 min |
| 10 · The long run | telephone lines, weather, $P^n$, stationary distributions | 20 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

Two very different problems turn out to need the same mathematics.

**A patient is coughing.** Is it bronchitis? Bronchitis is more likely in smokers and in people with a cold, and it
causes coughing — but so do other things. A physician reasons through a small web of causes and symptoms. Written as
a full joint distribution, five yes/no variables (smoking, cold, bronchitis, cough, asthma) need $2^5 - 1 = 31$
probabilities, and a realistic clinical or gene-regulatory model with 30 variables would need about a **billion**.
Nobody can estimate a billion numbers. **Bayesian networks** replace the giant table by a **graph** plus a few small
tables, one per variable, and answer questions such as "how likely is bronchitis in a smoker?" or "given a cough, how
likely is it that the patient smokes?".

**A stretch of DNA upstream of a gene.** Is it a **CpG island** — a region where the dinucleotide CG is *not*
depleted, typical of promoters — or ordinary background DNA? In [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb) you saw that neighbouring bases are
dependent. A **Markov chain** models exactly that dependence (each base depends on the previous one), and two Markov
chains, one for islands and one for background, let us score any sequence and scan a genome for islands — the idea
behind hidden-Markov-model gene finders.

| Biological question | Mathematical tool | Where |
|---|---|---|
| How can a model of many interacting variables stay small? | Bayesian network: $p(\mathbf x) = \prod_k p(x_k \mid \mathrm{Pa}_k)$ | §1–3 |
| Are two independent risk factors still independent among patients? | marginal independence of parents; explaining away | §4 |
| How likely is bronchitis in a smoker? Smoking given a cough? | inference by enumeration | §5 |
| Which variables must I measure to predict one node? | the Markov blanket | §7 |
| How do I model dependence between neighbouring bases? | Markov chain, transition matrix | §8 |
| Is this sequence a CpG island? Where are the islands? | likelihood ratio and log-odds scores of two chains | §9 |
| What happens after many steps? What is the long-run composition? | $P^n$ and the stationary distribution $\pi = \pi P$ | §10 |

> 🧪 **Simulated data, with a known truth.** The genome we scan contains a CpG island at a position we chose, so we can
> check whether the method finds it; every network probability is also checked by simulating patients from the network.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from itertools import product

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py). `draw_graph` draws small directed or undirected graphs with Matplotlib (we use it for every network), `simulate_dna` generates DNA from a Markov chain, `markov_log_probability` scores a sequence under a chain, and `CPG_ISLAND_TRANSITIONS` / `BACKGROUND_TRANSITIONS` are the two typed-in transition matrices of Section 9. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import matplotlib.pyplot as plt
import numpy as np


def circle_layout(nodes: list) -> dict:
    """Place nodes evenly on a circle (first node at the top, clockwise)."""
    n = len(nodes)
    angles = np.pi / 2 - 2 * np.pi * np.arange(n) / max(n, 1)
    return {v: (float(np.cos(a)), float(np.sin(a))) for v, a in zip(nodes, angles)}


def draw_graph(edges: list, nodes=None, directed: bool = False, pos=None, ax=None, node_colors=None,
               edge_labels=None, title: str | None = None, font_size: int = 9):
    """Draw a small graph with matplotlib only (no extra packages needed).

    ``edges`` is a list of (u, v) pairs; ``nodes`` (optional) adds isolated nodes and fixes the
    order; ``pos`` maps node → (x, y) (default: ``circle_layout``).  Directed edges get arrows,
    self-loops are drawn as small rings above the node, ``edge_labels`` maps (u, v) → text (e.g. a
    weight) and ``node_colors`` maps node → colour.  Node circles grow with the length of their
    label.  Returns the Axes.
    """
    from matplotlib.transforms import offset_copy

    if nodes is None:
        nodes = []
        for u, v in edges:
            for w in (u, v):
                if w not in nodes:
                    nodes.append(w)
    pos = circle_layout(list(nodes)) if pos is None else pos
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 3.6))
    radius = {v: max(0.62 * font_size * len(str(v)) / 2 + 4, 0.9 * font_size) for v in nodes}  # points
    for u, v in edges:
        if u == v:
            ring = offset_copy(ax.transData, fig=ax.figure, x=0, y=1.25 * radius[u], units="points")
            ax.scatter([pos[u][0]], [pos[u][1]], s=(1.3 * radius[u]) ** 2, facecolors="none",
                       edgecolors="0.35", linewidths=1.3, transform=ring, zorder=1)
            continue
        style = "-|>" if directed else "-"
        ax.annotate("", xy=pos[v], xytext=pos[u],
                    arrowprops=dict(arrowstyle=style, color="0.35", lw=1.3, mutation_scale=12,
                                    shrinkA=radius[u], shrinkB=radius[v],
                                    connectionstyle="arc3,rad=0.08" if directed else "arc3,rad=0"))
        if edge_labels and (u, v) in edge_labels:
            (x1, y1), (x2, y2) = pos[u], pos[v]
            ax.text((x1 + x2) / 2, (y1 + y2) / 2, str(edge_labels[(u, v)]), fontsize=font_size - 1,
                    ha="center", va="center", color="darkred", zorder=4,
                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))
    xs = [pos[v][0] for v in nodes]
    ys = [pos[v][1] for v in nodes]
    colours = [(node_colors or {}).get(v, "#cfe2f3") for v in nodes]
    ax.scatter(xs, ys, s=[(2 * radius[v]) ** 2 for v in nodes], c=colours, edgecolors="0.2",
               linewidths=1.2, zorder=2, clip_on=False)
    for v in nodes:
        ax.text(pos[v][0], pos[v][1], str(v), ha="center", va="center", fontsize=font_size, zorder=3,
                clip_on=False)
    ax.margins(0.22)
    ax.set_aspect("equal", adjustable="datalim")
    ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


NUCLEOTIDES = ("A", "C", "G", "T")


# First-order Markov-chain transition probabilities P(next | current) for DNA, rows = current base,
# columns = next base, both in the order A, C, G, T.  Rounded to two decimals from the classic
# CpG-island ("+") and background ("−") models of Durbin et al. (1998), Table 3.1, which were
# estimated from human sequence.  In CpG islands a C is often followed by a G; elsewhere CG is rare
# because methylated C in CpG mutates to T.
CPG_ISLAND_TRANSITIONS = np.array([
    [0.18, 0.27, 0.43, 0.12],
    [0.17, 0.37, 0.27, 0.19],
    [0.16, 0.34, 0.37, 0.13],
    [0.08, 0.36, 0.38, 0.18],
])


BACKGROUND_TRANSITIONS = np.array([
    [0.30, 0.20, 0.29, 0.21],
    [0.32, 0.30, 0.08, 0.30],
    [0.25, 0.25, 0.29, 0.21],
    [0.18, 0.24, 0.29, 0.29],
])


def simulate_dna(length: int, transitions=None, initial=None, seed: int = 0) -> str:
    """Random DNA string.  Without ``transitions`` the bases are i.i.d. uniform; otherwise they
    follow a first-order Markov chain with the given 4 × 4 transition matrix (order A, C, G, T)
    and initial distribution ``initial`` (default uniform)."""
    rng = np.random.default_rng(seed)
    initial = np.full(4, 0.25) if initial is None else np.asarray(initial, dtype=float)
    if transitions is None:
        return "".join(rng.choice(list(NUCLEOTIDES), size=length, p=initial))
    P = np.asarray(transitions, dtype=float)
    states = [rng.choice(4, p=initial)]
    for _ in range(length - 1):
        states.append(rng.choice(4, p=P[states[-1]]))
    return "".join(NUCLEOTIDES[s] for s in states)


def markov_log_probability(seq: str, transitions, initial=None) -> float:
    """Natural-log probability of a DNA string under a first-order Markov chain:
    log P(x1) + Σ log P(x_t | x_{t-1})."""
    idx = {b: i for i, b in enumerate(NUCLEOTIDES)}
    initial = np.full(4, 0.25) if initial is None else np.asarray(initial, dtype=float)
    P = np.asarray(transitions, dtype=float)
    logp = np.log(initial[idx[seq[0]]])
    for a, b in zip(seq[:-1], seq[1:]):
        logp += np.log(P[idx[a], idx[b]])
    return float(logp)

print("🧰 toolkit loaded:", ", ".join(sorted(['BACKGROUND_TRANSITIONS', 'CPG_ISLAND_TRANSITIONS', 'NUCLEOTIDES', 'circle_layout', 'draw_graph', 'markov_log_probability', 'simulate_dna'])))

In [ ]:
# ── 0.1 A SIMULATED genome stretch: background DNA with one CpG island hidden inside ──
left = simulate_dna(3000, BACKGROUND_TRANSITIONS, seed=11)
island = simulate_dna(1000, CPG_ISLAND_TRANSITIONS, seed=12)
right = simulate_dna(2000, BACKGROUND_TRANSITIONS, seed=13)
genome = left + island + right
island_start, island_end = len(left), len(left) + len(island)          # the hidden truth: positions 3000–3999


def gc_and_cpg(seq: str) -> tuple:
    """GC content and the observed/expected CG ratio P(CG) / (P(C) P(G)) of a sequence."""
    p_c, p_g = seq.count("C") / len(seq), seq.count("G") / len(seq)
    p_cg = sum(seq[k:k + 2] == "CG" for k in range(len(seq) - 1)) / (len(seq) - 1)
    return p_c + p_g, p_cg / (p_c * p_g)


print(f"genome length {len(genome):,} bp; true island at positions {island_start}–{island_end - 1}")
for name, seq in [("left background", left), ("island", island), ("right background", right)]:
    gc, oe = gc_and_cpg(seq)
    print(f"  {name:17s}: GC content {gc:.2f}, observed/expected CG {oe:.2f}")

🔍 **What just happened?** We glued 3,000 bp of background DNA, 1,000 bp of CpG-island DNA and 2,000 bp of background
DNA, all drawn from Markov chains (Section 8 explains how). The island has a higher GC content (0.67 versus 0.49 and
0.50) and a much higher observed/expected CG ratio (0.76 versus 0.34 and 0.28) than the background, exactly the signature biologists use to define CpG islands. In
Section 9 we will pretend we do not know where the island is and find it with the mathematics of this module.

---
## 1 · Why graphs? Bayesian networks as a language for joint distributions

> 📍 **Where we are.** [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb) described several random variables by their joint distribution — a table with one
> probability per combination of values. For $n$ binary variables the table has $2^n$ cells, so it needs $2^n - 1$
> free numbers (the last is fixed because they add up to 1). That is 31 for $n = 5$ and about $10^9$ for $n = 30$.
> This section introduces the graphical language that avoids the explosion.

A **Bayesian network**, also known as a **directed graphical model**, is a graphical language for specifying a
probabilistic model. Sometimes we need a more complicated conditional description than one formula, and a network
(a directed graph, [Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb)) is a way to write it. Bayesian networks **decompose a joint distribution into a product of
simpler conditional probabilities**, capturing the dependencies between variables and enabling more efficient
computation. They have some convenient properties:

* they are a simple way to **visualise the structure** of a probabilistic model;
* they can be used to **design or motivate new kinds of statistical models**;
* **inspection of the graph alone** gives insight into properties such as conditional independence;
* **complex computations** for inference and learning can be expressed as **graphical manipulations**.

**Nodes and arrows.** Each node is a random variable. A directed link (arrow) between two nodes indicates a
conditional probability: the arrow from $a$ to $b$ in the graph below gives the conditional probability $p(b \mid a)$
of $b$ given $a$. With additional assumptions the arrows can be used to indicate **causal** relationships (Pearl,
2009) — but a Bayesian network by itself is a statement about probabilities, not about causes.

In [ ]:
# ── 1.1 Two Bayesian networks: (a) fully connected, (b) not fully connected ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
draw_graph([("a", "b"), ("a", "c"), ("b", "c")], directed=True, ax=axes[0],
           pos={"a": (0, 1), "b": (2, 1), "c": (1, 0)}, title="(a) fully connected: a → b, a → c, b → c")
five_edges = [("x1", "x3"), ("x2", "x3"), ("x2", "x4"), ("x5", "x2")]
five_pos = {"x1": (0, 1), "x2": (2, 1), "x5": (4, 1), "x3": (1, 0), "x4": (3, 0)}
draw_graph(five_edges, nodes=["x1", "x2", "x3", "x4", "x5"], directed=True, ax=axes[1], pos=five_pos,
           title="(b) not fully connected: five variables, four arrows")
plt.tight_layout(); plt.show()
n_vars = np.array([3, 5, 10, 20, 30])
print("free parameters of a full joint table of n binary variables (2^n − 1):",
      dict(zip(n_vars.tolist(), (2 ** n_vars - 1).tolist())))

🔍 **What just happened?** Graph (a) connects every pair of its three variables; graph (b) has five variables but only
four arrows — many pairs, such as $x_1$ and $x_5$, are not linked directly. Missing arrows are what make a network
small: they are **assumptions** of conditional independence. The printed counts show why we need them: a full joint
table over 30 yes/no variables has 1,073,741,823 free parameters.

✏️ **Your turn.** In graph (b), list the **parents** (in-neighbours, [Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb)) and **children** (out-neighbours) of every
node. Which node has two parents? Which has none?

---
## 2 · From a factorisation to a graph — and back

> 📍 **Where we are.** The chain rule of [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb) writes any joint distribution as a product of conditionals. A Bayesian
> network is a picture of such a product. We now go both ways: product → picture and picture → product.

Consider the joint distribution of three random variables $a, b, c$ written as

$$p(a, b, c) = p(c \mid a, b)\,p(b \mid a)\,p(a).$$

*In words:* first choose $a$ from its own distribution, then $b$ from a distribution that depends on $a$, then $c$ from a
distribution that depends on both. This is just the chain rule — it holds for **every** joint distribution of three
variables. The factorisation tells us something about the relationship between the variables: **$c$ depends directly
on $a$ and $b$; $b$ depends directly on $a$; $a$ depends neither on $b$ nor on $c$** (its factor $p(a)$ has nothing
after the bar).

**Constructing the Bayesian network from a factorised joint distribution:**

1. **Create a node** for every random variable.
2. **For each conditional distribution, add a directed link (arrow)** to the graph *from* the nodes corresponding to the
   variables on which the distribution is conditioned *to* the variable whose distribution it is.

For $p(c \mid a, b)\,p(b \mid a)\,p(a)$: the factor $p(c \mid a, b)$ gives arrows $a \to c$ and $b \to c$; $p(b \mid a)$
gives $a \to b$; $p(a)$ gives no arrow. The result is graph (a) of Section 1.

In [ ]:
# ── 2.1 Build the graph of a factorisation; the same joint factorised in two orders ──
def graph_from_factorisation(factors: dict) -> list:
    """Edges of the Bayesian network of a factorisation {variable: [conditioning variables]}:
    one arrow from every conditioning variable to the variable of the factor."""
    return [(parent, child) for child, parents in factors.items() for parent in parents]


def factor_string(factors: dict) -> str:
    """Write a factorisation {variable: [parents]} as a product of p(·|·) terms."""
    return " ".join(f"p({v} | {', '.join(ps)})" if ps else f"p({v})" for v, ps in factors.items())


order_abc = {"a": [], "b": ["a"], "c": ["a", "b"]}       # p(a) p(b | a) p(c | a, b)
order_cba = {"c": [], "b": ["c"], "a": ["b", "c"]}       # p(c) p(b | c) p(a | b, c)
chain_abc = {"a": [], "b": ["a"], "c": ["b"]}            # p(a) p(b | a) p(c | b): one arrow fewer

layout = {"a": (0, 1), "b": (2, 1), "c": (1, 0)}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
for ax, factors in zip(axes, [order_abc, order_cba, chain_abc]):
    draw_graph(graph_from_factorisation(factors), nodes=["a", "b", "c"], directed=True, pos=layout, ax=ax,
               title=factor_string(factors))
plt.tight_layout(); plt.show()
for factors in [order_abc, order_cba, chain_abc]:
    print(f"{factor_string(factors):32s} → edges {graph_from_factorisation(factors)}")

🔍 **What just happened?** The function turned each factorisation into its list of arrows. The first two factorisations
are both the chain rule — one in the order $a, b, c$, the other in the order $c, b, a$ — so both are correct for **any**
joint distribution of three variables, yet their graphs point in opposite directions. **The graph layout depends on the
choice of factorisation of the joint distribution**: a joint distribution does not have a unique network. The third
graph has only two arrows: it claims $p(c \mid a, b) = p(c \mid b)$, i.e. $c$ is independent of $a$ *given* $b$ — an
assumption that is true for some distributions and false for others. Let us check both statements numerically on a
random joint distribution.

In [ ]:
# ── 2.2 Check: both chain-rule factorisations reproduce a random joint; the chain p(a)p(b|a)p(c|b) does not ──
rng = np.random.default_rng(SEED + 22)
joint_abc = rng.dirichlet(np.ones(8)).reshape(2, 2, 2)      # a random joint p(a, b, c), axes (a, b, c)
p_a = joint_abc.sum(axis=(1, 2)); p_b = joint_abc.sum(axis=(0, 2)); p_c = joint_abc.sum(axis=(0, 1))
p_ab = joint_abc.sum(axis=2); p_bc = joint_abc.sum(axis=0)
p_b_given_a = p_ab / p_a[:, None]                           # p(b | a), rows a
p_c_given_ab = joint_abc / p_ab[:, :, None]                 # p(c | a, b)
p_b_given_c = p_bc / p_c[None, :]                           # p(b | c), columns c
p_a_given_bc = joint_abc / p_bc[None, :, :]                 # p(a | b, c)
p_c_given_b = p_bc / p_b[:, None]                           # p(c | b), rows b
rebuilt_abc = p_a[:, None, None] * p_b_given_a[:, :, None] * p_c_given_ab
rebuilt_cba = p_c[None, None, :] * p_b_given_c[None, :, :] * p_a_given_bc
rebuilt_chain = p_a[:, None, None] * p_b_given_a[:, :, None] * p_c_given_b[None, :, :]
for name, rebuilt in [("p(a) p(b|a) p(c|a,b)", rebuilt_abc), ("p(c) p(b|c) p(a|b,c)", rebuilt_cba),
                      ("p(a) p(b|a) p(c|b)  ", rebuilt_chain)]:
    print(f"{name}: largest |product − joint| = {np.abs(rebuilt - joint_abc).max():.2e}")

🔍 **What just happened?** Both chain-rule products rebuild the random joint distribution up to floating-point rounding
(differences 0 and $3 \times 10^{-18}$): every complete ordering is valid. The two-arrow chain misses by 0.016 — a clearly
non-zero amount — because this random distribution does **not** satisfy $c \perp a \mid b$. Drawing a network with missing arrows is therefore a
modelling decision that must be justified by knowledge of the system (for example, "a gene's expression depends on its
transcription factors, not directly on the upstream signal").

### 2.2 Reading the joint distribution off a graph

Now the reverse process: derive the joint distribution from a graph. Take graph (b) of Section 1. We exploit two
properties:

1. The joint distribution $p(x_1, \dots, x_5)$ is the **product of a set of conditionals, one for each node** in the
   graph — here we need five conditionals.
2. Each conditional **depends only on the parents** of the corresponding node; for example $x_4$ will be conditioned on
   $x_2$.

These two properties yield the factorisation

$$p(x_1, x_2, x_3, x_4, x_5) = p(x_1)\,p(x_5)\,p(x_2 \mid x_5)\,p(x_3 \mid x_1, x_2)\,p(x_4 \mid x_2).$$

*In words:* $x_1$ and $x_5$ have no parents, so they get unconditional factors; $x_2$ has the parent $x_5$; $x_3$ has
the two parents $x_1, x_2$; $x_4$ has the parent $x_2$. In general, the joint distribution $p(\mathbf x) = p(x_1,
\dots, x_K)$ of a Bayesian network is

$$p(\mathbf x) = \prod_{k=1}^{K} p(x_k \mid \mathrm{Pa}_k),$$

*In words:* multiply, over all $K$ nodes, the probability of the node given its parents, where $\mathrm{Pa}_k$ means
"the parent nodes of $x_k$" — the nodes that have arrows pointing to $x_k$. In the graph language of [Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb) the
parents are the **in-neighbours**, $\mathrm{Pa}_k = N^{in}(x_k)$, so the same formula can be written for variables
$Q_1, \dots, Q_d$ as

$$p\big(\langle Q_1, Q_2, \dots, Q_d\rangle\big) = \prod_{i=1}^{d} p\Big(Q_i \,\Big|\, \textstyle\bigwedge_{j \in N^{in}(Q_i)} Q_j\Big),$$

*In words:* the probability of the whole tuple of values is the product, over the $d$ variables (indices $i = 1$ to
$d$), of each variable's probability given the conjunction ("and", $\wedge$) of its in-neighbours. A node without
in-neighbours contributes an unconditional factor.

In [ ]:
# ── 2.3 Read the factorisation and count parameters directly from a graph ──
def parents_of(node, edges):
    """In-neighbours N_in(node): the tails of all arrows that point to the node."""
    return [u for (u, v) in edges if v == node]


def factorisation_from_graph(nodes, edges) -> dict:
    """{node: parents} — one conditional per node, conditioned on its parents."""
    return {v: parents_of(v, edges) for v in nodes}


def n_parameters(factors: dict, n_values: int = 2) -> int:
    """Free parameters of a network of variables with n_values values each:
    (n_values − 1) numbers for every combination of parent values."""
    return sum((n_values - 1) * n_values ** len(ps) for ps in factors.values())


five_factors = factorisation_from_graph(["x1", "x5", "x2", "x3", "x4"], five_edges)
print("p(x1, ..., x5) =", factor_string(five_factors))
print("free parameters, binary variables: network", n_parameters(five_factors), "vs full joint", 2 ** 5 - 1)
for v, ps in five_factors.items():
    print(f"  {v}: parents {ps or '—'} → {2 ** len(ps)} parameter(s)")

🔍 **What just happened?** Reading parents off the edge list reproduced the factorisation
$p(x_1)\,p(x_5)\,p(x_2 \mid x_5)\,p(x_3 \mid x_1, x_2)\,p(x_4 \mid x_2)$. If all five variables are binary, each factor
needs one number per combination of its parents' values: $1 + 1 + 2 + 4 + 2 = 10$ free parameters instead of $31$.
The saving grows exponentially with the number of variables, as long as every node has few parents.

⚠️ **Acyclic only.** A Bayesian network must be a **directed acyclic graph (DAG)**: following arrows must never bring
you back to where you started. With a cycle $a \to b \to a$ the "product of conditionals" $p(a \mid b)\,p(b \mid a)$ is
not, in general, a joint distribution at all. Because there is no cycle, the nodes can always be listed so that parents
come before children (a **topological order**, here $x_1, x_5, x_2, x_3, x_4$) — the order in which we will simulate
networks.

✏️ **Your turn.** (1) Write the factorisation of the graph with arrows $x_1 \to x_2$, $x_1 \to x_3$, $x_2 \to x_4$,
$x_3 \to x_4$ and count its parameters for binary variables. (2) Draw (with `draw_graph`) the network of
$p(x_1)\,p(x_2 \mid x_1)\,p(x_3 \mid x_2)\,p(x_4 \mid x_3)$. What is this shape called? (Section 8 answers.)

### 🌉 Bridge — Bayesian networks of genes and signalling proteins

In 02-620, Bayesian networks are the language for **gene regulatory networks** and **signalling pathways**: each node
is a gene or protein, each arrow means "its level depends directly on", and the factorisation $\prod_k p(x_k \mid
\mathrm{Pa}_k)$ turns a network of 11 signalling proteins (the classic flow-cytometry study of Sachs *et al.*, 2005) from
a table with hundreds of thousands of entries into a few hundred numbers ([BiologyToML 15 · Bayesian networks: representation](../../../../ml/tutorial/BiologyToML/notebooks/15_bayesian_networks_representation.ipynb)). The next modules of that course
read conditional independences off the graph with **d-separation** ([BiologyToML 16 · Bayesian networks: independence](../../../../ml/tutorial/BiologyToML/notebooks/16_bayesian_networks_independence.ipynb)) and estimate the tables — and even the
arrows — from data ([BiologyToML 17 · Bayesian networks: inference and learning](../../../../ml/tutorial/BiologyToML/notebooks/17_bayesian_networks_inference_learning.ipynb)). Everything starts from the two-way translation you just practised.

---
## 3 · Numbers on a network: conditional probability tables

> 📍 **Where we are.** A graph alone says *which* variables depend on which. To compute probabilities we attach to
> every node a **conditional probability table (CPT)**: the distribution of the node for every combination of its
> parents' values.

Consider binary variables $X, Y, Z$ (values 0 and 1) with arrows $X \to Y$, $Y \to Z$ and $X \to Z$ — the fully connected
graph (a) of Section 1, now with numbers. Its factorisation is $p(x, y, z) = p(x)\,p(y \mid x)\,p(z \mid x, y)$, and we specify (illustrative
numbers):

| $P(X = 1)$ | | $X$ | $P(Y = 1 \mid X)$ | | $X$ | $Y$ | $P(Z = 1 \mid X, Y)$ |
|---|---|---|---|---|---|---|---|
| 0.3 | | 0 | 0.5 | | 0 | 0 | 0.3 |
| | | 1 | 0.9 | | 0 | 1 | 0.1 |
| | | | | | 1 | 0 | 0.7 |
| | | | | | 1 | 1 | 0.4 |

(so $P(X = 0) = 0.7$, $P(Y = 0 \mid X = 0) = 0.5$, $P(Y = 0 \mid X = 1) = 0.1$, and so on — each row's two probabilities
add up to 1). If we alter the network to remove the dependence of $Z$ on $X$, we get the **chain** $X \to Y \to Z$ with
$p(x, y, z) = p(x)\,p(y \mid x)\,p(z \mid y)$ and a smaller table for $Z$:

| $Y$ | $P(Z = 1 \mid Y)$ |
|---|---|
| 0 | 0.2 |
| 1 | 0.7 |

We store a network as a dictionary `node: (parents, {parent values: P(node = 1 | parent values)})`, listed in a
topological order, and write three small functions: the full joint table, a query by enumeration, and simulation.

In [ ]:
# ── 3.1 Bayesian networks of binary variables: joint table, queries, simulation ──
full_xyz = {"X": ((), {(): 0.3}),
            "Y": (("X",), {(0,): 0.5, (1,): 0.9}),
            "Z": (("X", "Y"), {(0, 0): 0.3, (0, 1): 0.1, (1, 0): 0.7, (1, 1): 0.4})}
chain_xyz = {"X": ((), {(): 0.3}),
             "Y": (("X",), {(0,): 0.5, (1,): 0.9}),
             "Z": (("Y",), {(0,): 0.2, (1,): 0.7})}


def network_joint(network: dict) -> pd.DataFrame:
    """Full joint table of a binary Bayesian network: one row per configuration, p = Π p(node | parents)."""
    nodes = list(network)
    rows = []
    for values in product([0, 1], repeat=len(nodes)):
        config = dict(zip(nodes, values))
        prob = 1.0
        for node, (parents, cpt) in network.items():
            p1 = cpt[tuple(config[q] for q in parents)]          # P(node = 1 | its parents' values)
            prob *= p1 if config[node] == 1 else 1 - p1
        rows.append({**config, "p": prob})
    return pd.DataFrame(rows)


def query(joint: pd.DataFrame, target: str, evidence: dict | None = None) -> float:
    """P(target = 1 | evidence) by enumeration: sum the joint over matching rows, then divide."""
    match = np.ones(len(joint), dtype=bool)
    for var, val in (evidence or {}).items():
        match &= joint[var].to_numpy() == val
    return joint.loc[match & (joint[target] == 1), "p"].sum() / joint.loc[match, "p"].sum()


def sample_network(network: dict, n: int, rng) -> pd.DataFrame:
    """Ancestral sampling: draw every node after its parents (the dict is in topological order)."""
    data = {}
    for node, (parents, cpt) in network.items():
        keys = zip(*(data[q] for q in parents)) if parents else [()] * n
        p1 = np.array([cpt[tuple(int(v) for v in k)] for k in keys])
        data[node] = (rng.random(n) < p1).astype(int)
    return pd.DataFrame(data)


joint_full = network_joint(full_xyz)
print(joint_full.assign(p=joint_full.p.round(4)).to_string(index=False))
print("sum of the joint:", round(joint_full.p.sum(), 12))
print(f"P(Y = 1) = 0.7·0.5 + 0.3·0.9 = {query(joint_full, 'Y'):.3f};   P(Z = 1) = {query(joint_full, 'Z'):.3f}")

🔍 **What just happened?** `network_joint` multiplied the three CPT entries for each of the $2^3 = 8$ configurations;
e.g. the first row is $p(0, 0, 0) = 0.7 \times 0.5 \times 0.7 = 0.245$. The eight probabilities add up to 1, as they must.
From the joint we can answer any question by **enumeration** — summing the rows that match: $P(Y = 1) = 0.7 \cdot 0.5 +
0.3 \cdot 0.9 = 0.62$ and $P(Z = 1) = 0.105 + 0.035 + 0.021 + 0.108 = 0.269$. Now the chain, and the conditional
independence it encodes.

In [ ]:
# ── 3.2 The chain X → Y → Z: Z ⊥ X | Y holds; in the full network it does not ──
joint_chain = network_joint(chain_xyz)
for name, joint in [("full network (X → Z present)", joint_full), ("chain X → Y → Z", joint_chain)]:
    print(f"── {name}:  P(Z = 1) = {query(joint, 'Z'):.3f}")
    for yv in (0, 1):
        vals = [query(joint, "Z", {"X": xv, "Y": yv}) for xv in (0, 1)]
        print(f"   P(Z = 1 | X = 0, Y = {yv}) = {vals[0]:.2f},  P(Z = 1 | X = 1, Y = {yv}) = {vals[1]:.2f}")
    print(f"   without conditioning on Y: P(Z = 1 | X = 0) = {query(joint, 'Z', {'X': 0}):.3f},"
          f"  P(Z = 1 | X = 1) = {query(joint, 'Z', {'X': 1}):.3f}")
print("free parameters: full network", 1 + 2 + 4, "(= 2³ − 1, no saving) | chain", 1 + 2 + 2)

rng = np.random.default_rng(SEED + 32)
sim = sample_network(chain_xyz, 100_000, rng)
print(f"100,000 SIMULATED draws from the chain: P(Z = 1) ≈ {sim.Z.mean():.3f}, "
      f"P(Z = 1 | X = 1, Y = 1) ≈ {sim.Z[(sim.X == 1) & (sim.Y == 1)].mean():.3f}")

🔍 **What just happened?** In the **chain**, once $Y$ is fixed, $X$ makes no difference: $P(Z = 1 \mid X, Y = 0) = 0.20$
and $P(Z = 1 \mid X, Y = 1) = 0.70$ for both values of $X$ — **$Z$ is conditionally independent of $X$ given $Y$**,
because $Z$'s table only looks at $Y$. Yet $X$ and $Z$ are *not* independent: $P(Z = 1 \mid X = 1) = 0.65$ but
$P(Z = 1 \mid X = 0) = 0.45$ — the influence of $X$ travels through $Y$. In the **full network**, conditioning on $Y$ does
not remove the influence of $X$ ($0.30$ vs $0.70$ when $Y = 0$). The full network has $1 + 2 + 4 = 7$ parameters —
exactly $2^3 - 1$, because a fully connected network makes no independence assumption — while the chain needs 5. The
100,000 simulated draws (each variable drawn after its parents) reproduce $P(Z = 1) = 0.51$ and $P(Z = 1 \mid X = 1, Y = 1)
= 0.70$ to two decimals.

✏️ **Your turn.** (1) Compute $P(Z = 1)$ for the chain by hand: $P(Y = 0)\,P(Z = 1 \mid Y = 0) + P(Y = 1)\,P(Z = 1 \mid
Y = 1)$. (2) Use `query` to compute $P(X = 1 \mid Z = 1)$ in both networks — "diagnostic" reasoning against the arrows.

### 📘 Lecture notes — Bayesian networks: definition and factorisation

**Bayesian network (directed graphical model).** A directed acyclic graph whose nodes are random variables, plus a
conditional distribution for each node given its parents. It provides a compact way to specify a probabilistic model,
lets us visually parse dependencies, motivates new models, reveals conditional independences by inspection and turns
inference and learning into graph manipulations. An arrow $a \to b$ stands for $p(b \mid a)$; with additional
assumptions arrows may be read causally (Pearl, 2009).

**Joint distribution of a network.**

$$p(\mathbf x) = \prod_{k=1}^{K} p(x_k \mid \mathrm{Pa}_k), \qquad \mathrm{Pa}_k = N^{in}(x_k) \text{ (the in-neighbours of } x_k).$$

*In words:* one factor per node, each conditioned only on the node's parents.

**Factorisation → graph.** (1) Create a node for every variable; (2) for each conditional, draw arrows from the
conditioning variables to the variable of the factor. $p(a, b, c) = p(c \mid a, b)\,p(b \mid a)\,p(a)$: $c$ depends
directly on $a$ and $b$, $b$ on $a$, $a$ on neither. The graph depends on the chosen factorisation (order).

**Graph → factorisation (five-node example).** Arrows $x_1 \to x_3$, $x_2 \to x_3$, $x_2 \to x_4$, $x_5 \to x_2$:
$p(x_1, \dots, x_5) = p(x_1)\,p(x_5)\,p(x_2 \mid x_5)\,p(x_3 \mid x_1, x_2)\,p(x_4 \mid x_2)$.

**CPT example.** $P(X = 1) = 0.3$; $P(Y = 1 \mid X) = 0.5, 0.9$; $P(Z = 1 \mid X, Y) = 0.3, 0.1, 0.7, 0.4$ for
$(X, Y) = (0,0), (0,1), (1,0), (1,1)$. Removing $X \to Z$ gives the chain with $P(Z = 1 \mid Y) = 0.2, 0.7$ and
$Z \perp X \mid Y$.

**Parameters (binary variables).** Full joint: $2^n - 1$. Network: $\sum_k 2^{|\mathrm{Pa}_k|}$.

> 🎓 **Exam tip.** When asked for "the factorisation implied by the graph", write exactly one factor per node — no more,
> no fewer — and double-check each node's parents by looking for arrow *heads* pointing at it.

---
## 4 · Two causes, one effect: marginal independence and explaining away

> 📍 **Where we are.** The bronchitis network of Section 5 has two arrows pointing into one node (smoking → bronchitis ←
> cold). Before using it, we prove what such a "v-structure" implies about the two parents.

The classic example: an alarm $A$ can be set off by a burglary $B$ or by an earthquake $E$ — two causes with no common
ancestor ($B \to A \leftarrow E$). (A biological analogue: two unrelated risk factors — a germline variant and an
environmental exposure — that can each trigger the same disease.) The factorisation is $p(b, e, a) = p(b)\,p(e)\,p(a
\mid b, e)$. Are $B$ and $E$ independent? Sum $A$ out:

$$\begin{aligned}
P(B = b, E = e) &= \sum_a P(B = b, E = e, A = a) \\
&= \sum_a p(b)\,p(e)\,p(a \mid b, e) \\
&= p(b)\,p(e)\sum_a p(a \mid b, e) \\
&= p(b)\,p(e), \qquad\text{because } \sum_a p(a \mid b, e) = 1.
\end{aligned}$$

*In words:* line 1 marginalises the joint ([Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb)); line 2 inserts the network's factorisation; line 3 moves the
factors that do not depend on $a$ out of the sum; line 4 uses the fact that a conditional distribution sums to 1 over
its own variable. So **the two parents are marginally independent**. ⚠️ But they are **not** independent *given* $A$:
if the alarm rang and we learn there was an earthquake, a burglary becomes *less* likely — the earthquake "explains
away" the alarm.

In [ ]:
# ── 4.1 B → A ← E: independent parents that become dependent once A is observed (ILLUSTRATIVE numbers) ──
alarm_net = {"B": ((), {(): 0.01}),
             "E": ((), {(): 0.02}),
             "A": (("B", "E"), {(0, 0): 0.001, (0, 1): 0.29, (1, 0): 0.94, (1, 1): 0.95})}
joint_alarm = network_joint(alarm_net)
p_BE = joint_alarm.groupby(["B", "E"]).p.sum()                         # sum A out
print("P(B, E) from the joint vs P(B) P(E):")
for (bv, ev), val in p_BE.items():
    pb = 0.01 if bv else 0.99
    pe = 0.02 if ev else 0.98
    print(f"  B={bv}, E={ev}: {val:.6f}  vs  {pb * pe:.6f}")
print(f"P(B = 1) = {query(joint_alarm, 'B'):.4f};  P(B = 1 | E = 1) = {query(joint_alarm, 'B', {'E': 1}):.4f}  → independent")
print(f"P(B = 1 | A = 1) = {query(joint_alarm, 'B', {'A': 1}):.4f};  "
      f"P(B = 1 | A = 1, E = 1) = {query(joint_alarm, 'B', {'A': 1, 'E': 1}):.4f}  → explaining away")

🔍 **What just happened?** Every cell of $P(B, E)$ equals $P(B)\,P(E)$ to all printed digits, and $P(B = 1 \mid E = 1) =
P(B = 1) = 0.01$: without information about the alarm, the two causes are independent, as the derivation promised. Once
the alarm is known to have rung, the picture changes: $P(B = 1 \mid A = 1) \approx 0.58$, but if we also learn that
there was an earthquake it drops to $P(B = 1 \mid A = 1, E = 1) \approx 0.03$. Observing a common effect makes its
causes **dependent**. This is why a study that recruits only patients who already have a disease can find a spurious
negative association between two unrelated risk factors (selection bias, the "collider" of [BiologyToML 16 · Bayesian networks: independence](../../../../ml/tutorial/BiologyToML/notebooks/16_bayesian_networks_independence.ipynb)).

---
## 5 · The bronchitis network

> 📍 **Where we are.** Everything so far, on a network a physician would recognise. We derive one probability by hand —
> naming the rule used in every line — and then let the computer enumerate the rest.

The network below models the relationships among **bronchitis** and its potential causes and symptoms. Its Boolean
variables represent whether the person **smokes** ($S$), has a **cold** ($C$), has **bronchitis** ($T$), is **coughing**
($O$) and has **asthma** ($A$). Arrows: $S \to T$, $C \to T$, $T \to O$, $T \to A$. The conditional probabilities
(illustrative numbers; $\neg$ means "not"):

| node | conditional probability table |
|---|---|
| $S$ | $P(S) = 0.4$, $P(\neg S) = 0.6$ |
| $C$ | $P(C) = 0.8$, $P(\neg C) = 0.2$ |
| $T$ | $P(T \mid S, C) = 0.35$, $P(T \mid \neg S, C) = 0.25$, $P(T \mid S, \neg C) = 0.1$, $P(T \mid \neg S, \neg C) = 0.05$ |
| $O$ | $P(O \mid T) = 0.6$, $P(O \mid \neg T) = 0.15$ |
| $A$ | $P(A \mid T) = 0.5$, $P(A \mid \neg T) = 0.1$ |

The factorisation is $p(s, c, t, o, a) = p(s)\,p(c)\,p(t \mid s, c)\,p(o \mid t)\,p(a \mid t)$ — 10 parameters instead
of 31.

**Q: Given that a person smokes, what is the probability that they have bronchitis?** Step by step:

$$\begin{aligned}
P(T \mid S) &= P(T, C \mid S) + P(T, \neg C \mid S) && \text{(sum rule: } C \text{ is either true or false)}\\[2pt]
P(T, C \mid S) &= \frac{P(T, C, S)}{P(S)} && \text{(definition of conditional probability)}\\
&= \frac{P(T \mid C, S)\,P(C, S)}{P(S)} && \text{(product rule)}\\
&= P(T \mid C, S)\,P(C \mid S) && \text{(definition again: } P(C, S)/P(S) = P(C \mid S))\\
&= P(T \mid C, S)\,P(C) && (C \text{ and } S \text{ are independent — two parents, Section 4})
\end{aligned}$$

*In words:* we split on the unknown cold status, rewrite each piece with the definition of conditional probability
and the product rule, and finally use the fact that smoking and having a cold are independent in this network (they
are two parents with no common ancestor). Likewise $P(T, \neg C \mid S) = P(T \mid \neg C, S)\,P(\neg C)$, so

$$P(T \mid S) = P(T \mid C, S)\,P(C) + P(T \mid \neg C, S)\,P(\neg C) = 0.35 \times 0.8 + 0.1 \times 0.2 = 0.28 + 0.02 = 0.30.$$

*In words:* a smoker's bronchitis risk is the average of the risks with and without a cold, weighted by how common
colds are. ⚠️ The step from $P(T, C, S)/P(S)$ to $P(T \mid C, S)P(C, S)/P(S)$ is sometimes labelled "Bayes"; it is the
product rule (Bayes' rule is the product rule written both ways, [Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb)).

In [ ]:
# ── 5.1 The bronchitis network: draw it, enumerate the joint, answer queries, check by simulation ──
bronchitis = {"S": ((), {(): 0.4}),
              "C": ((), {(): 0.8}),
              "T": (("S", "C"), {(1, 1): 0.35, (0, 1): 0.25, (1, 0): 0.10, (0, 0): 0.05}),
              "O": (("T",), {(1,): 0.6, (0,): 0.15}),
              "A": (("T",), {(1,): 0.5, (0,): 0.10})}
bron_edges = [(q, v) for v, (ps, _) in bronchitis.items() for q in ps]
bron_pos = {"S": (0, 2), "C": (2, 2), "T": (1, 1), "O": (0, 0), "A": (2, 0)}
fig, ax = plt.subplots(figsize=(3.6, 3.2))
draw_graph(bron_edges, nodes=list(bronchitis), directed=True, pos=bron_pos, ax=ax,
           title="S smokes, C cold, T bronchitis,\nO cough, A asthma")
plt.tight_layout(); plt.show()

joint_bron = network_joint(bronchitis)
print("32 configurations; sum of the joint =", round(joint_bron.p.sum(), 12))
print(f"P(T | S)  by hand: 0.35·0.8 + 0.1·0.2 = {0.35 * 0.8 + 0.1 * 0.2:.2f};  by enumeration: {query(joint_bron, 'T', {'S': 1}):.4f}")
questions = [("T", {}, "P(T)        bronchitis, no information"), ("O", {}, "P(O)        cough"),
             ("T", {"O": 1}, "P(T | O)    bronchitis given a cough"), ("S", {"O": 1}, "P(S | O)    smoker given a cough"),
             ("S", {"T": 1}, "P(S | T)    smoker given bronchitis"), ("C", {"T": 1}, "P(C | T)    cold given bronchitis"),
             ("S", {"T": 1, "C": 0}, "P(S | T, ¬C) smoker given bronchitis without a cold")]
rng = np.random.default_rng(SEED + 51)
patients = sample_network(bronchitis, 200_000, rng)                  # SIMULATED patients
for target, ev, label in questions:
    sel = np.ones(len(patients), dtype=bool)
    for var, val in ev.items():
        sel &= patients[var].to_numpy() == val
    print(f"{label:52s} exact {query(joint_bron, target, ev):.4f} | 200,000 simulated: {patients.loc[sel, target].mean():.4f}")

🔍 **What just happened?** Enumeration over all 32 configurations confirms the hand calculation $P(T \mid S) = 0.30$.
Without any information, $P(T) = 0.4 \cdot 0.8 \cdot 0.35 + 0.6 \cdot 0.8 \cdot 0.25 + 0.4 \cdot 0.2 \cdot 0.1 + 0.6 \cdot
0.2 \cdot 0.05 = 0.246$ and the probability of a cough is $P(O) = 0.246 \cdot 0.6 + 0.754 \cdot 0.15 = 0.261$. Evidence
flows **against** the arrows too: a cough raises the probability of bronchitis from 0.246 to about 0.57 and the
probability that the patient smokes from 0.40 to about 0.44 (smoking raises bronchitis, bronchitis raises coughing).
Learning that a bronchitis patient has **no** cold raises the probability that they smoke (from 0.49 to 0.57) —
explaining away again: without the cold, smoking is the remaining explanation. The 200,000 simulated patients match six of the seven
exact answers to within 0.002; the largest deviation (0.562 vs 0.571) is in the rarest group, bronchitis without a cold,
which contains only about 2,800 of the simulated patients, so its estimate is the noisiest.

✏️ **Your turn.** (1) Compute $P(T \mid \neg S)$ by hand with the same steps and check with `query`. (2) What is
$P(O \mid S)$? (*Hint:* $P(O \mid S) = P(O \mid T)P(T \mid S) + P(O \mid \neg T)P(\neg T \mid S)$ — which independence
justifies this?) (3) Does observing asthma change the probability of a cough? Compare $P(O)$ and $P(O \mid A)$.

### 📘 Lecture notes — computing with a Bayesian network

**Marginal independence of parents** ($B \to A \leftarrow E$):
$P(B = b, E = e) = \sum_a p(b)\,p(e)\,p(a \mid b, e) = p(b)\,p(e)\sum_a p(a \mid b, e) = p(b)\,p(e)$, since
$\sum_a p(a \mid b, e) = 1$. Given the common child they become dependent (explaining away).

**Bronchitis network.** $S$ (smokes, 0.4), $C$ (cold, 0.8) → $T$ (bronchitis: $P(T \mid S, C) = 0.35$,
$P(T \mid \neg S, C) = 0.25$, $P(T \mid S, \neg C) = 0.1$, $P(T \mid \neg S, \neg C) = 0.05$) → $O$ (cough 0.6 / 0.15),
$A$ (asthma 0.5 / 0.1).

$$P(T \mid S) = P(T, C \mid S) + P(T, \neg C \mid S) = P(T \mid C, S)P(C) + P(T \mid \neg C, S)P(\neg C) = 0.35 \cdot 0.8 + 0.1 \cdot 0.2 = 0.30,$$

*In words:* split on the unobserved cold, use the definition of conditional probability and the product rule, then the
independence of $C$ and $S$ to replace $P(C \mid S)$ by $P(C)$. Further results: $P(T) = 0.246$, $P(O) = 0.2607$.

**Inference by enumeration.** $P(Q \mid \text{evidence}) = \dfrac{\sum_{\text{rows matching } Q, \text{evidence}} p}{\sum_{\text{rows matching evidence}} p}$
— always correct, but the number of rows grows as $2^n$; real systems use smarter algorithms (variable elimination,
belief propagation) or sampling.

> 🎓 **Exam tip.** Label every line of a derivation with the rule you used (sum rule, definition, product rule, independence
> *and why*). The independence step is where marks are lost: say *which* variables are independent and *which part of the
> graph* guarantees it.

---
## 6 · Three kinds of graphical model

> 📍 **Where we are.** Bayesian networks use arrows. Two other graphical languages describe joint distributions with
> undirected edges; you will meet both in 02-620.

There are three types of graphical models:

* **(a) Directed graphical models (Bayesian networks)** — arrows, one conditional distribution per node.
* **(b) Undirected graphical models (Markov random fields, Markov networks)** — plain edges; the joint distribution is a
  normalised product of non-negative **potential functions** $\psi_C$, one for each **clique** $C$ (a set of nodes that
  are all connected to each other, [Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb)):

$$p(\mathbf x) = \frac{1}{Z}\prod_{C \in \mathcal C}\psi_C(\mathbf x_C), \qquad Z = \sum_{\mathbf x}\prod_{C \in \mathcal C}\psi_C(\mathbf x_C).$$

*In words:* each clique gets a "compatibility score" for the values of its variables; multiply the scores and divide by
their total $Z$ (the **partition function**) so that probabilities add up to 1. The defining property is

$$p(x_i \mid \mathbf x_{-i}) = p\big(x_i \mid \mathbf x_{N(X_i)}\big),$$

*In words:* given its **neighbours** $N(X_i)$, a node is independent of all the other nodes ($\mathbf x_{-i}$ means
"every variable except $x_i$").

* **(c) Factor graphs** — a bipartite graph ([Module 06 · 🕸️ Graphs and trees: networks and phylogenies](06_graphs_and_trees.ipynb)) with circles for variables and squares for factors, which makes
  the product structure explicit; both (a) and (b) can be converted into factor graphs, which is how many inference
  algorithms work.

In [ ]:
# ── 6.1 Draw the three types of graphical model and an 8-node Markov network ──
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), gridspec_kw={"width_ratios": [1, 1, 1, 2.2]})
pos3 = {"a": (0, 1), "b": (2, 1), "c": (1, 0)}
draw_graph([("a", "c"), ("b", "c")], directed=True, pos=pos3, ax=axes[0], title="(a) directed graphical model")
draw_graph([("a", "c"), ("b", "c")], directed=False, pos=pos3, ax=axes[1], title="(b) undirected graphical model")
pos_f = {"a": (0, 1), "b": (2, 1), "c": (1, 0), "f": (1, 1)}
draw_graph([("a", "f"), ("b", "f"), ("c", "f")], directed=False, pos=pos_f, ax=axes[2], title="(c) factor graph")
axes[2].scatter([1], [1], marker="s", s=330, color="k", zorder=5)            # the factor node is a black square
axes[2].text(1, 1, "f", color="white", ha="center", va="center", fontsize=9, zorder=6)
mrf_edges = [("X1", "X2"), ("X1", "X3"), ("X1", "X4"), ("X2", "X3"), ("X2", "X4"), ("X3", "X4"),
             ("X2", "X5"), ("X5", "X6"), ("X6", "X7"), ("X6", "X8"), ("X7", "X8")]
mrf_pos = {"X1": (0, 1), "X2": (2, 1), "X3": (0, 0), "X4": (2, 0), "X5": (3.5, 1), "X6": (5, 1),
           "X7": (7, 1), "X8": (6, 0)}
draw_graph(mrf_edges, directed=False, pos=mrf_pos, ax=axes[3],
           title="a Markov network: p(xᵢ | x₋ᵢ) = p(xᵢ | neighbours)")
plt.tight_layout(); plt.show()
nbrs = {v: sorted({u for e in mrf_edges for u in e if v in e} - {v}) for v in mrf_pos}
print("neighbours N(X5) =", nbrs["X5"], "| N(X2) =", nbrs["X2"], "| N(X8) =", nbrs["X8"])

🔍 **What just happened?** The three small graphs show the same three variables in the three languages: arrows into $c$
(a); plain edges (b); and a factor graph in which one square factor $f(a, b, c)$ is joined to all three variables (c).
The Markov network on the right has the cliques $\{X_1, X_2, X_3, X_4\}$ (every pair connected), $\{X_6, X_7, X_8\}$ and
the edges $\{X_2, X_5\}$, $\{X_5, X_6\}$; its joint distribution would be $\frac1Z\psi_{1234}\,\psi_{25}\,\psi_{56}\,
\psi_{678}$. Node $X_5$ has only two neighbours, $X_2$ and $X_6$: given those two, it is independent of the other five
variables. Let us verify the defining property on the smallest Markov network, the path $a - c - b$.

In [ ]:
# ── 6.2 A tiny Markov network a — c — b: potentials, partition function Z, and a ⊥ b | c ──
psi_ac = np.array([[3.0, 1.0], [1.0, 2.0]])          # compatibility of (a, c): ILLUSTRATIVE, need not sum to 1
psi_cb = np.array([[2.0, 1.0], [1.0, 4.0]])          # compatibility of (c, b)
unnorm = np.einsum("ac,cb->acb", psi_ac, psi_cb)     # product of the potentials for every (a, c, b)
Z = unnorm.sum()
p_acb = unnorm / Z
print(f"partition function Z = {Z}; probabilities sum to {p_acb.sum():.3f}")
for cv in (0, 1):
    p_a1_given_cb = [p_acb[1, cv, bv] / p_acb[:, cv, bv].sum() for bv in (0, 1)]
    print(f"c = {cv}: P(a = 1 | c, b = 0) = {p_a1_given_cb[0]:.4f},  P(a = 1 | c, b = 1) = {p_a1_given_cb[1]:.4f}")
p_ab = p_acb.sum(axis=1)
print(f"without c: P(a = 1 | b = 0) = {p_ab[1, 0] / p_ab[:, 0].sum():.4f},  P(a = 1 | b = 1) = {p_ab[1, 1] / p_ab[:, 1].sum():.4f}")

🔍 **What just happened?** The potentials are arbitrary positive scores, not probabilities; dividing their product by
$Z$ (here 27) turns it into a joint distribution. Given $c$, the probability of $a$ does not depend on $b$
($P(a = 1 \mid c = 0, b) = 0.25$ and $P(a = 1 \mid c = 1, b) = 0.6667$ for both values of $b$) — the neighbour $c$ shields
$a$ from $b$. Without $c$, $a$ and $b$ are dependent ($P(a = 1 \mid b = 0) = 0.36$ but $P(a = 1 \mid b = 1) = 0.56$). In a Bayesian network the corresponding "shield" is slightly larger,
because co-parents matter too: the **Markov blanket**.

### 🌉 Bridge — undirected models of genes: Gaussian graphical models

When the variables are continuous and jointly Gaussian ([Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb), Section 8), the Markov random field is a **Gaussian
graphical model**: an edge between genes $i$ and $j$ is present exactly when the $(i, j)$ entry of the inverse covariance
matrix is non-zero, i.e. when the two genes are dependent *given all other genes*. Estimating that sparse inverse
(graphical lasso) is a standard way to infer co-expression networks from expression data ([BiologyToML 18 · Gaussian graphical models](../../../../ml/tutorial/BiologyToML/notebooks/18_gaussian_graphical_models.ipynb)). Factor graphs are
the data structure behind belief propagation, the inference algorithm of [BiologyToML 17 · Bayesian networks: inference and learning](../../../../ml/tutorial/BiologyToML/notebooks/17_bayesian_networks_inference_learning.ipynb).

---
## 7 · The Markov blanket

> 📍 **Where we are.** To predict one node of a big network, do we need to measure every other node? No — only its
> Markov blanket.

The **Markov blanket** of a node $X$ in a Bayesian network is the minimal set of nodes that renders $X$ conditionally
independent of all other nodes in the network. Given its Markov blanket, a node is conditionally independent of
everything else:

$$P\big(X \mid \mathrm{MB}(X), \text{Rest}\big) = P\big(X \mid \mathrm{MB}(X)\big).$$

*In words:* once you know the blanket, learning anything about the rest of the network does not change your belief
about $X$. For a node $X$ the Markov blanket consists of:

* the **parents** of $X$ — nodes with arrows pointing to $X$;
* the **children** of $X$ — nodes with arrows from $X$;
* the **co-parents** of $X$ — the other parents of $X$'s children.

Why the co-parents? Because a child depends on *all* its parents, so the child's factor $p(\text{child} \mid X,
\text{co-parent})$ ties $X$ to the co-parent — exactly the explaining-away effect of Section 4. **Example.** In the
network with arrows $V_{11} \to U_1$, $V_{12} \to U_1$, $U_1 \to X_2$, $U_1 \to X$ and $U_2 \to X$:

$$P(U_1 \mid \text{Rest}) = P(U_1 \mid V_{11}, V_{12}, X_2, X, U_2), \qquad P(U_2 \mid \text{Rest}) = P(U_2 \mid X, U_1).$$

*In words:* the blanket of $U_1$ contains its parents $V_{11}, V_{12}$, its children $X_2, X$ and the co-parent $U_2$
(the other parent of $X$); the blanket of $U_2$ contains its child $X$ and the co-parent $U_1$ — not $V_{11}$, $V_{12}$ or
$X_2$.

In [ ]:
# ── 7.1 Find Markov blankets from the edge list; draw the example ──
def markov_blanket(node, edges) -> set:
    """Parents ∪ children ∪ co-parents (other parents of the node's children)."""
    parents = {u for (u, v) in edges if v == node}
    children = {v for (u, v) in edges if u == node}
    co_parents = {u for (u, v) in edges if v in children and u != node}
    return parents | children | co_parents


mb_edges = [("V11", "U1"), ("V12", "U1"), ("U1", "X2"), ("U1", "X"), ("U2", "X")]
mb_pos = {"V11": (0, 2), "V12": (2, 2), "U1": (1, 1), "U2": (3, 1), "X2": (0, 0), "X": (2, 0)}
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, target in zip(axes, ["U1", "U2"]):
    blanket = markov_blanket(target, mb_edges)
    colours = {v: ("0.55" if v == target else "#fdae6b" if v in blanket else "#cfe2f3") for v in mb_pos}
    draw_graph(mb_edges, nodes=list(mb_pos), directed=True, pos=mb_pos, ax=ax, node_colors=colours, font_size=8,
               title=f"MB({target}) = {sorted(blanket)} (orange)")
plt.tight_layout(); plt.show()
for target in mb_pos:
    print(f"MB({target:3s}) = {sorted(markov_blanket(target, mb_edges))}")
print("bronchitis network: MB(S) =", sorted(markov_blanket("S", bron_edges)),
      "| MB(T) =", sorted(markov_blanket("T", bron_edges)), "| MB(O) =", sorted(markov_blanket("O", bron_edges)))

🔍 **What just happened?** The function reproduces the two blankets of the example: $\mathrm{MB}(U_1) = \{U_2, V_{11},
V_{12}, X, X_2\}$ (in this small network that is every other node) and $\mathrm{MB}(U_2) = \{U_1, X\}$. Leaves have small
blankets: $\mathrm{MB}(X_2) = \{U_1\}$, its only parent. In the bronchitis network, the blanket of smoking is
$\{C, T\}$ — its child bronchitis and bronchitis's other parent, cold — while cough and asthma are outside it. Now we
**verify** the defining property numerically: give the network random CPTs, compute the full joint, and check that
$P(U_2 \mid \text{everything else})$ depends only on the blanket.

In [ ]:
# ── 7.2 Numerical check: P(node | all other nodes) depends only on the Markov blanket ──
def random_network(edges, order, rng) -> dict:
    """A binary Bayesian network with the given arrows and random CPT entries in (0.05, 0.95)."""
    net = {}
    for v in order:
        ps = tuple(u for (u, w) in edges if w == v)
        net[v] = (ps, {k: rng.uniform(0.05, 0.95) for k in product([0, 1], repeat=len(ps))})
    return net


def full_conditional_check(joint: pd.DataFrame, target: str, blanket: set) -> pd.DataFrame:
    """P(target = 1 | all other variables) for every configuration, grouped by the blanket's values:
    if the blanket shields the target, every group contains a single distinct value."""
    others = [c for c in joint.columns if c not in (target, "p")]
    wide = joint.pivot_table(index=others, columns=target, values="p")
    full_cond = (wide[1] / (wide[0] + wide[1])).rename("P(target=1 | rest)").reset_index()
    grouped = full_cond.groupby(sorted(blanket))["P(target=1 | rest)"]
    return pd.DataFrame({"value": grouped.mean(), "spread": grouped.max() - grouped.min(), "configs": grouped.size()})


rng = np.random.default_rng(SEED + 72)
mb_net = random_network(mb_edges, ["V11", "V12", "U2", "U1", "X2", "X"], rng)
joint_mb = network_joint(mb_net)
print("P(U2 = 1 | the other 5 nodes), grouped by the blanket {U1, X}:")
print(full_conditional_check(joint_mb, "U2", markov_blanket("U2", mb_edges)).round(4))
print("\nbronchitis network — P(S = 1 | C, T, O, A), grouped by the blanket {C, T}:")
print(full_conditional_check(joint_bron, "S", markov_blanket("S", bron_edges)).round(4))

🔍 **What just happened?** For each of the four combinations of the blanket $\{U_1, X\}$ there are 8 configurations of
the remaining nodes $V_{11}, V_{12}, X_2$, and all 8 give the **same** $P(U_2 = 1 \mid \text{rest})$ — the "spread" column
is 0. Changing the non-blanket nodes does nothing once the blanket is known. In the bronchitis network, likewise,
$P(S = 1 \mid C, T, O, A)$ takes only four distinct values, one per combination of the blanket $\{C, T\}$; cough and asthma
add nothing once bronchitis and cold are known. Notice the values: with bronchitis present ($T = 1$), a smoker is more
likely when there is **no** cold (0.571) than when there is one (0.483) — the co-parent matters, which is why it
belongs to the blanket.

✏️ **Your turn.** (1) Remove $U_2$ from the blanket in `full_conditional_check(joint_mb, "U2", {"X"})`. Is the spread
still 0? Why not? (2) Find the Markov blanket of $x_2$ in the five-node graph of Section 2 and check it with a random
network.

### 🌉 Bridge — Markov blankets in feature selection and sampling

If a disease status is a node in a network of genes, its Markov blanket is the **smallest set of genes that predicts it
as well as all the genes together** — the ideal target for feature selection, and the reason Markov-blanket discovery
algorithms are used to pick biomarkers. The blanket is also what makes **Gibbs sampling** cheap: to update one variable
given all others, a sampler only needs the variable's blanket ([BiologyToML 21 · Bayesian learning and MCMC](../../../../ml/tutorial/BiologyToML/notebooks/21_bayesian_learning_mcmc.ipynb)). In 02-620 the blanket is derived from the
d-separation rules ([BiologyToML 16 · Bayesian networks: independence](../../../../ml/tutorial/BiologyToML/notebooks/16_bayesian_networks_independence.ipynb)): conditioning on parents and children blocks every path out of the node, and conditioning
on a child opens the path to the co-parents, which must therefore be included too.

### 📘 Lecture notes — three types of graphical models and the Markov blanket

**Types.** (a) Directed graphical models (Bayesian networks): $p(\mathbf x) = \prod_k p(x_k \mid \mathrm{Pa}_k)$.
(b) Undirected graphical models (Markov random fields / Markov networks): $p(\mathbf x) = \frac1Z\prod_C \psi_C(\mathbf
x_C)$ over cliques, with the property $p(x_i \mid \mathbf x_{-i}) = p(x_i \mid \mathbf x_{N(X_i)})$. (c) Factor graphs:
bipartite graphs of variables (circles) and factors (squares).

**Markov blanket.** The minimal set of nodes that renders $X$ conditionally independent of all other nodes:

$$P(X \mid \mathrm{MB}(X), \text{Rest}) = P(X \mid \mathrm{MB}(X)), \qquad \mathrm{MB}(X) = \text{parents} \cup \text{children} \cup \text{co-parents}.$$

*In words:* given its parents, its children and its children's other parents, a node carries no further information
about the rest of the network.

**Example.** $V_{11} \to U_1 \leftarrow V_{12}$, $U_1 \to X_2$, $U_1 \to X \leftarrow U_2$:
$P(U_1 \mid \text{Rest}) = P(U_1 \mid V_{11}, V_{12}, X_2, X, U_2)$; $P(U_2 \mid \text{Rest}) = P(U_2 \mid X, U_1)$.
In a Markov network the blanket is simply the set of neighbours.

> 🎓 **Exam tip.** The co-parents are the most frequently forgotten part of a Markov blanket. Find them by looking at
> every child and listing *its* other parents.

---
## 8 · Markov chains: sequences with a short memory

> 📍 **Where we are.** A sequence — of DNA letters, of daily weather, of a cell's states over time — is a set of random
> variables with a natural order. A **Markov chain** is the simplest Bayesian network for such a sequence: each
> variable has one parent, the previous one.

A **Markov chain** is a special type of **stochastic process** (a sequence of random variables $X_1, X_2, \dots$ indexed
by time or position), defined in terms of the conditional distributions of future states given the present and past
states. It is a probability distribution for a "sequence" of random variables that models dependencies among
observations in **adjacent** positions.

**Definition.** A stochastic process with a discrete time parameter is a Markov chain if, for each time $n$, the
conditional distributions of all $X_{n+j}$ ($j \ge 1$) given $X_1, \dots, X_n$ depend only on $X_n$ and not on the earlier
states $X_1, \dots, X_{n-1}$. In symbols, for $n = 1, 2, \dots$, for each $b$ and each possible sequence of states $x_1,
x_2, \dots, x_n$:

$$\Pr(X_{n+1} \le b \mid X_1 = x_1, X_2 = x_2, \dots, X_n = x_n) = \Pr(X_{n+1} \le b \mid X_n = x_n).$$

*In words:* to predict the next state, the whole history is no more useful than the current state. A Markov chain is
called **finite** if there are only finitely many possible states (4 for DNA, 2 for "sunny/cloudy").

**The joint distribution.** Markov chains are usually used to model sequences of items. From conditional probability
([Module 17 · 🔀 Conditional probability, independence and Bayes' rule](17_conditional_probability_bayes.ipynb)) the chain rule always gives

$$\begin{aligned}
p(X_n, X_{n-1}, \dots, X_1) &= p(X_n \mid X_{n-1}, \dots, X_1)\,p(X_{n-1}, \dots, X_1) \\
&= p(X_n \mid X_{n-1}, \dots, X_1)\,p(X_{n-1} \mid X_{n-2}, \dots, X_1)\,p(X_{n-2}, \dots, X_1) \\
&\;\;\vdots \\
&= p(X_n \mid X_{n-1}, \dots, X_1)\cdots p(X_2 \mid X_1)\,p(X_1).
\end{aligned}$$

*In words:* peel off the last variable again and again with the product rule until only $p(X_1)$ is left. The **Markov
assumption** is that the probability of $X_k$ depends on $X_{k-1}$ alone. This simplifies the computation, at the cost
of long-range connections. Anecdotally the Markov assumption says: **the future is independent of the past given the
present.** So we can rewrite the statement above as

$$p(X_n, X_{n-1}, \dots, X_1) = p(X_n \mid X_{n-1})\,p(X_{n-1} \mid X_{n-2})\cdots p(X_2 \mid X_1)\,p(X_1).$$

*In words:* the probability of a whole sequence is the probability of its first state times one **transition
probability** for every step. For a finite chain this is the joint probability function of the first $n$ states,
$\Pr(X_1 = x_1, \dots, X_n = x_n) = \Pr(X_1 = x_1)\Pr(X_2 = x_2 \mid X_1 = x_1)\cdots\Pr(X_n = x_n \mid X_{n-1} = x_{n-1})$.
This is also the basis for **hidden Markov models**, which are used a lot in genetics.

**Transition matrix.** If the transition probabilities do not change along the sequence ("stationary transition
probabilities"), they fit in a matrix $P$ with entries $P_{ij} = \Pr(X_{t+1} = j \mid X_t = i)$: **rows = current state,
columns = next state**, every row adding up to 1.

In [ ]:
# ── 8.1 The chain as a Bayesian network, a two-state transition diagram, and simulation ──
fig, axes = plt.subplots(1, 2, figsize=(11, 2.8), gridspec_kw={"width_ratios": [1.8, 1]})
chain_nodes = ["X1", "X2", "X3", "…", "Xn−1", "Xn"]
draw_graph(list(zip(chain_nodes[:-1], chain_nodes[1:])), nodes=chain_nodes, directed=True,
           pos={v: (i, 0) for i, v in enumerate(chain_nodes)}, ax=axes[0], font_size=8,
           title="a Markov chain as a Bayesian network: each variable's only parent is its predecessor")
weather_P = np.array([[0.7, 0.3], [0.6, 0.4]])            # rows: today sunny/cloudy; columns: tomorrow
draw_graph([("sunny", "cloudy"), ("cloudy", "sunny"), ("sunny", "sunny"), ("cloudy", "cloudy")], directed=True,
           pos={"sunny": (0, 0), "cloudy": (2, 0)}, ax=axes[1], font_size=8, title="weather chain: transition diagram")
for txt, xy in [("0.6", (1, 0.22)), ("0.3", (1, -0.22)), ("0.7", (0, 0.55)), ("0.4", (2, 0.55))]:
    axes[1].text(*xy, txt, ha="center", fontsize=8, color="darkred")
plt.tight_layout(); plt.show()


def simulate_chain(P, initial, n_steps: int, rng) -> np.ndarray:
    """States X_1, ..., X_n of a Markov chain with transition matrix P and initial distribution `initial`."""
    states = [rng.choice(len(initial), p=initial)]
    for _ in range(n_steps - 1):
        states.append(rng.choice(P.shape[1], p=P[states[-1]]))   # next state drawn from the current state's row
    return np.array(states)


print("rows of the weather matrix sum to", weather_P.sum(axis=1))
rng = np.random.default_rng(SEED + 81)
days = simulate_chain(weather_P, [1.0, 0.0], 14, rng)
print("two SIMULATED weeks, starting sunny:", " ".join("S" if d == 0 else "c" for d in days))
print(f"P(S, S, c, c) = P(X1 = S)·P(S→S)·P(S→c)·P(c→c) = 1 · 0.7 · 0.3 · 0.4 = {1 * 0.7 * 0.3 * 0.4:.3f}")
print("free parameters for DNA of length 10: full joint 4^10 − 1 =", 4 ** 10 - 1, "| Markov chain 3 + 4·3 =", 3 + 4 * 3)

🔍 **What just happened?** The left graph is the Bayesian network of a Markov chain — a path in which every variable has
exactly one parent, its predecessor; reading off its factorisation (Section 2) gives exactly the Markov formula above.
The right diagram is a different picture of the same model, the **transition diagram**: nodes are the *states* (sunny,
cloudy), arrows are labelled with transition probabilities, and the probabilities leaving each state add up to 1
($0.7 + 0.3$ and $0.6 + 0.4$). The simulated fortnight shows the chain's "stickiness": sunny days tend to follow sunny
days. The probability of a whole sequence is a product of transitions, e.g. $1 \cdot 0.7 \cdot 0.3 \cdot 0.4 = 0.084$
for sunny, sunny, cloudy, cloudy starting from a sunny day. For DNA, a first-order Markov chain that uses the same transition matrix at every step needs 15 numbers (3 for
the first letter, 3 for each of the 4 rows) instead of $4^{10} - 1 = 1{,}048{,}575$ for a full table of 10 letters.

---
## 9 · Application: is this DNA a CpG island?

> 📍 **Where we are.** Now a real use of Markov chains — modelling a sequence of nucleotides.

Consider a sequence of DNA nucleotides modelled as a set of random variables $N_1 N_2 \dots N_n$ (more generally a
sequence of length $T$, $X_1, \dots, X_T$), each of which takes a value in $\{A, T, C, G\}$. We want to know if the
sequence is a CpG island or not, and the object we need is the **joint distribution of all random variables**
$P(X_1, \dots, X_T)$. Two cases:

* **Case 1 — background genome, simplest model.** The letters are independent, each with its own base composition
  $p(x)$, so the joint distribution is a product of single-letter probabilities,

$$P(X_1, \dots, X_T) = \prod_{t=1}^{T} p(x_t).$$

*In words:* with independent letters, the probability of a sequence is the product of the probabilities of its letters
— for uniform composition, simply $(1/4)^T$. This model cannot see CpG depletion, which is a property of *pairs* of
neighbours.

* **Case 2 — CpG islands versus non-CpG DNA.** The dinucleotide CG is easily mutated to TG (a methylated C deaminates to
  T), so CG is rare in most of the genome. **CpG islands** are areas of the genome where the CG sequence is enriched,
  often found around promoters or the start regions of genes. To capture "C followed by G" we need dependence between
  neighbours: a **Markov chain** for each kind of DNA.

$P_b$ and $P_c$ are two different transition probability matrices: $P_b$ for the **background** model (normal, non-CpG-
island DNA) and $P_c$ for the **CpG-island** model. Each gives the probability of transitioning from one nucleotide to
the next — **rows = current nucleotide, columns = next nucleotide**. Looking at the Markov factorisation, if we know the
initial distribution $p(N_1)$ for both cases and the transition distributions $p(N_k \mid N_{k-1})$, we can determine
which model makes the sequence more probable:

$$P(N_1, \dots, N_n \mid \text{model}) = p(N_1)\prod_{k=2}^{n} P_{\text{model}}(N_{k-1} \to N_k).$$

*In words:* the probability of the sequence under a model is the probability of its first letter times the transition
probability of every neighbouring pair. **Question:** assuming for both models $p(N_1 = A) = p(N_1 = C) = p(N_1 = T) =
p(N_1 = G) = 0.25$, is the sequence **ACTTC** more likely to be a CpG island or not?

In [ ]:
# ── 9.1 The two transition matrices and the sequence ACTTC ──
P_c, P_b = CPG_ISLAND_TRANSITIONS, BACKGROUND_TRANSITIONS
for name, M in [("P_c (CpG island)", P_c), ("P_b (background)", P_b)]:
    print(name, "— rows = current base, columns = next base")
    print(pd.DataFrame(M, index=list(NUCLEOTIDES), columns=list(NUCLEOTIDES)).to_string(), "\nrow sums:", M.sum(axis=1))
idx = {b: i for i, b in enumerate(NUCLEOTIDES)}


def sequence_probability(seq: str, P, initial=0.25) -> float:
    """p(N1) · Π P(N_{k-1} → N_k) for a first-order Markov chain with a uniform first letter."""
    prob = initial
    for a, b in zip(seq[:-1], seq[1:]):
        prob *= P[idx[a], idx[b]]
    return prob


seq = "ACTTC"
factors_c = " · ".join(f"{P_c[idx[a], idx[b]]:.2f}" for a, b in zip(seq[:-1], seq[1:]))
factors_b = " · ".join(f"{P_b[idx[a], idx[b]]:.2f}" for a, b in zip(seq[:-1], seq[1:]))
p_cpg, p_bg = sequence_probability(seq, P_c), sequence_probability(seq, P_b)
print(f"\nP(ACTTC | CpG)     = P(A)·P(C|A)·P(T|C)·P(T|T)·P(C|T) = 0.25 · {factors_c} = {p_cpg:.8f}")
print(f"P(ACTTC | non-CpG) = 0.25 · {factors_b} = {p_bg:.6f}")
print(f"ratio P(·|non-CpG) / P(·|CpG) = {p_bg / p_cpg:.3f}")
print("check with the toolkit's markov_log_probability:",
      round(np.exp(markov_log_probability(seq, P_c)), 8), round(np.exp(markov_log_probability(seq, P_b)), 6))

🔍 **What just happened?** Both matrices have rows that sum to 1. The most striking difference is the entry C → G:
$0.27$ in islands but only $0.08$ in background DNA — CpG depletion written as a transition probability. For ACTTC,

* $P(\text{ACTTC} \mid \text{CpG}) = P(A)\,P(C \mid A)\,P(T \mid C)\,P(T \mid T)\,P(C \mid T) = 0.25 \times 0.27 \times 0.19
  \times 0.18 \times 0.36 = 0.00083106$;
* $P(\text{ACTTC} \mid \text{non-CpG}) = 0.25 \times 0.20 \times 0.30 \times 0.29 \times 0.24 = 0.001044$;
* $P(\text{ACTTC} \mid \text{non-CpG}) / P(\text{ACTTC} \mid \text{CpG}) = 0.001044 / 0.00083106 \approx 1.26 > 1$.

Hence ACTTC is **more likely under the non-CpG (background) model**. The ratio is modest — five letters carry little
evidence, and ACTTC contains no CG at all. The toolkit function, which adds logarithms instead of multiplying, gives the
same probabilities.

### 9.2 Log-odds scores and scanning a genome

Products of many probabilities below 1 quickly underflow to 0 on a computer, so we work with **logarithms**. The
**log-odds score** of a sequence $x$ is

$$S(x) = \log\frac{P(x \mid \text{CpG})}{P(x \mid \text{background})} = \sum_{k=2}^{n}\log\frac{(P_c)_{x_{k-1}x_k}}{(P_b)_{x_{k-1}x_k}} \quad (\text{the equal first-letter terms cancel}).$$

*In words:* every neighbouring pair contributes its own score — positive if the pair is more typical of islands,
negative if more typical of background — and the scores simply add up. $S > 0$ means "island more likely". For ACTTC,
$S = \log(1/1.256) = -0.23$. We move a window of 200 bp along the simulated genome of Section 0.1 and compute $S$ in each
window.

In [ ]:
# ── 9.2 Per-pair log-odds, sliding-window scores along the SIMULATED genome ──
log_odds = np.log(P_c / P_b)
print("log-odds log(P_c / P_b) per pair (rows = current base, columns = next base):")
print(pd.DataFrame(log_odds, index=list(NUCLEOTIDES), columns=list(NUCLEOTIDES)).round(3))

codes = np.array([idx[b] for b in genome])
pair_scores = log_odds[codes[:-1], codes[1:]]                # score of every neighbouring pair (k−1, k)
window = 200
window_scores = np.convolve(pair_scores, np.ones(window - 1), mode="valid")   # sum over the 199 pairs of a window
centres = np.arange(len(window_scores)) + window / 2
called = window_scores > 0
inside = (centres >= island_start) & (centres < island_end)
print(f"\n{len(window_scores):,} windows of {window} bp; mean score inside the island {window_scores[inside].mean():.1f}, "
      f"outside {window_scores[~inside].mean():.1f}")
print(f"windows called 'island' (S > 0): {called[inside].mean():.1%} of windows centred in the island, "
      f"{called[~inside].mean():.1%} of the others ({called[~inside].sum()} of {(~inside).sum():,})")
first, last = centres[called].min(), centres[called].max()
print(f"called region: window centres {first:.0f}–{last:.0f}  (true island {island_start}–{island_end - 1})")

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.axvspan(island_start, island_end, color="tab:orange", alpha=0.25, label="true CpG island (SIMULATED)")
ax.plot(centres, window_scores, color="tab:blue", lw=1, label="log-odds score of a 200-bp window")
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("position in the genome (bp)"); ax.set_ylabel("S = log P(CpG) / P(background)")
ax.set_title("scanning a genome with two Markov chains"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The largest per-pair score is C → G, $\log(0.27/0.08) = 1.216$: a CG pair is the strongest
single piece of evidence for an island, while pairs such as C → A or T → A count against it. Summed over 199 pairs, the
window score is strongly negative in background DNA and strongly positive in the island, so the curve rises above zero
almost exactly where the true island is (orange band): 97.4 % of the windows centred in the island are called and only 1 of the
4,801 others (0.0 %), and the called window centres run from 2994 to 3981 against a true island at 3000–3999. Near the
boundaries the score passes through zero gradually, because a window that straddles a boundary contains a mixture of
both kinds of DNA. This is the first step towards a **hidden Markov
model**, which lets the hidden label "island / background" itself follow a Markov chain and finds the boundaries
automatically ([BiologyToML 19 · Hidden Markov models: inference](../../../../ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb)).

✏️ **Your turn.** (1) Change `window` to 50 and to 500. How do the false calls and the sharpness of the boundaries
change? Explain the trade-off. (2) Compute the log-odds of the sequence CGCGCG by hand from the table and check it with
`markov_log_probability`.

### 📘 Lecture notes — Markov chains and CpG islands

**Markov chain.** A stochastic process with
$\Pr(X_{n+1} \le b \mid X_1 = x_1, \dots, X_n = x_n) = \Pr(X_{n+1} \le b \mid X_n = x_n)$: the future is independent of
the past given the present. Finite if there are finitely many states. Transition matrix $P_{ij} = \Pr(X_{t+1} = j \mid
X_t = i)$, rows sum to 1.

**Factorisation.** Chain rule $p(X_n, \dots, X_1) = p(X_n \mid X_{n-1}, \dots, X_1)\cdots p(X_2 \mid X_1)\,p(X_1)$; with
the Markov assumption

$$p(X_n, \dots, X_1) = p(X_n \mid X_{n-1})\,p(X_{n-1} \mid X_{n-2})\cdots p(X_2 \mid X_1)\,p(X_1).$$

*In words:* first state times one transition per step.

**CpG islands.** CG mutates easily to TG; islands are CG-enriched regions near promoters. Background matrix $P_b$ (rows
A, C, G, T: 0.30 0.20 0.29 0.21 / 0.32 0.30 0.08 0.30 / 0.25 0.25 0.29 0.21 / 0.18 0.24 0.29 0.29) and island matrix
$P_c$ (0.18 0.27 0.43 0.12 / 0.17 0.37 0.27 0.19 / 0.16 0.34 0.37 0.13 / 0.08 0.36 0.38 0.18). With a uniform first
letter: $P(\text{ACTTC} \mid \text{CpG}) = 0.25 \cdot 0.27 \cdot 0.19 \cdot 0.18 \cdot 0.36 = 0.00083106$,
$P(\text{ACTTC} \mid \text{non-CpG}) = 0.25 \cdot 0.20 \cdot 0.30 \cdot 0.29 \cdot 0.24 = 0.001044$; ratio
$0.001044 / 0.00083106 \approx 1.26 > 1$ ⇒ more likely non-CpG. Log-odds scores add over pairs.

**Common confusions.** In $P_{ij}$ the row is the *current* state; rows (not columns) sum to 1. Independent letters
(Case 1) cannot model CpG depletion.

> 🎓 **Exam tip.** Write the product with every factor labelled ($P(C \mid A)$, $P(T \mid C)$, …) before plugging in
> numbers; most errors come from reading the matrix by column instead of by row.

---
## 10 · Many steps ahead: $P^n$, telephone lines, the weather, and the long run

> 📍 **Where we are.** Markov chains answer questions about the future: where will the process be in 3 steps? in the
> long run? The answers are **matrix powers** ([Module 08 · 🔢 Matrices: tables you can multiply](08_matrices.ipynb)) and **eigenvectors** ([Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)).

**Two-step transitions.** To go from state $i$ to state $j$ in two steps, the chain passes through some intermediate
state $k$. By the sum rule and the Markov property,

$$\Pr(X_{t+2} = j \mid X_t = i) = \sum_{k}\Pr(X_{t+1} = k \mid X_t = i)\,\Pr(X_{t+2} = j \mid X_{t+1} = k) = \sum_k P_{ik}P_{kj} = (P^2)_{ij}.$$

*In words:* add up, over all intermediate states, the probability of the path $i \to k \to j$; that sum is exactly the
row-times-column rule of matrix multiplication. Repeating the argument, the **$n$-step transition probabilities** are the
entries of $P^n$, and $P^{m+n} = P^m P^n$ (the **Chapman–Kolmogorov equations**). If the row vector $\mathbf v$ holds the
distribution of the state at one time, the distribution one step later is $\mathbf v P$, and $n$ steps later
$\mathbf v P^n$.

**Example (telephone lines).** A business office has five telephone lines, any number of which may be in use at a given
time. The lines are observed at regular intervals of 2 minutes and the number in use is noted; $X_n$ is the number of
lines in use at the $n$-th observation (states 0–5). Suppose the initial probability vector is $\mathbf v = (0.5, 0.3,
0.2, 0, 0, 0)$, with $v_i = \Pr(X_1 = i)$, and the transition matrix is (rows = lines in use now, columns = at the next
observation):

$$P = \begin{bmatrix}
0.1 & 0.4 & 0.2 & 0.1 & 0.1 & 0.1\\
0.2 & 0.3 & 0.2 & 0.1 & 0.1 & 0.1\\
0.1 & 0.2 & 0.3 & 0.2 & 0.1 & 0.1\\
0.1 & 0.1 & 0.2 & 0.3 & 0.2 & 0.1\\
0.1 & 0.1 & 0.1 & 0.2 & 0.3 & 0.2\\
0.1 & 0.1 & 0.1 & 0.1 & 0.4 & 0.2
\end{bmatrix}.$$

*In words:* each row is the distribution of the next count given the current count; e.g. with all five lines busy now,
the next observation finds 4 busy with probability 0.4. **Q1:** what is the probability that **no** lines will be in use
three time periods later if all five lines are in use at a certain time? **Q2:** what is the distribution of $X_3$?

In [ ]:
# ── 10.1 Telephone lines: P², P³, a 3-step probability and the distribution of X3 (typed-in example) ──
P_tel = np.array([[0.1, 0.4, 0.2, 0.1, 0.1, 0.1],
                  [0.2, 0.3, 0.2, 0.1, 0.1, 0.1],
                  [0.1, 0.2, 0.3, 0.2, 0.1, 0.1],
                  [0.1, 0.1, 0.2, 0.3, 0.2, 0.1],
                  [0.1, 0.1, 0.1, 0.2, 0.3, 0.2],
                  [0.1, 0.1, 0.1, 0.1, 0.4, 0.2]])
v = np.array([0.5, 0.3, 0.2, 0, 0, 0])                      # distribution of X1
P2 = P_tel @ P_tel
P3 = P2 @ P_tel
print("row sums of P:", P_tel.sum(axis=1))
print("P² =\n", P2.round(2))
print("P³ =\n", P3.round(3))
print(f"Q1: P(X_(t+3) = 0 | X_t = 5) = (P³)[5, 0] = {P3[5, 0]:.3f}")
print("    by hand, two-step entry (P²)[5, 0] = Σ_k P[5, k] P[k, 0] =",
      " + ".join(f"{P_tel[5, k]:.1f}·{P_tel[k, 0]:.1f}" for k in range(6)), f"= {P2[5, 0]:.2f}")
print("Q2: distribution of X3 = v P² =", (v @ P2).round(3), " (X1 has distribution v, X3 is two steps later)")
print("    distribution of X4 = v P³ =", (v @ P3).round(3))

🔍 **What just happened?** The matrix powers reproduce the classic tables: $P^2$ (rounded to two decimals) and $P^3$ (to
three). **Q1:** the probability that no lines will be in use three periods after all five are busy is the entry in row 5,
column 0 of $P^3$: **0.116**. The hand computation of the two-step entry $(P^2)_{5,0} = 0.1\cdot0.1 + 0.1\cdot0.2 + \dots
= 0.11$ shows the "sum over intermediate states" at work. **Q2:** $\mathbf v$ is the distribution of $X_1$, so $X_3$ —
two steps later — has distribution $\mathbf v P P = \mathbf v P^2 = (0.133, 0.227, 0.202, 0.156, 0.162, 0.120)$.
⚠️ **Mind the indexing.** Because the chain starts at $X_1$ (not $X_0$), "the distribution of $X_3$" needs $P^2$; the
distribution of $X_4$ would need $P^3$. Always count the number of *steps*, not the subscript.

**Example (weather).** Suppose the weather can only be sunny or cloudy, and the weather conditions on successive
mornings form a Markov chain with stationary transition probabilities, with the matrix of Section 8 (sunny → sunny 0.7,
sunny → cloudy 0.3, cloudy → sunny 0.6, cloudy → cloudy 0.4). **Q:** if it is sunny on a certain Wednesday, what is the
probability that it will be sunny on **both** the following Saturday and Sunday? Saturday is three days after
Wednesday, so we first need $(P^3)_{\text{sunny},\text{sunny}}$; then Sunday must follow Saturday with one more sunny →
sunny step:

$$\Pr(\text{Sat} = S, \text{Sun} = S \mid \text{Wed} = S) = (P^3)_{SS}\cdot P_{SS} = 0.667 \times 0.7 \approx 0.467.$$

*In words:* the Markov factorisation of the joint event "Saturday sunny and Sunday sunny" is "Saturday sunny (a 3-step
probability) times Sunday sunny given Saturday sunny (a 1-step probability)".

In [ ]:
# ── 10.2 Weather: P³, the two-day question, and a simulation check (typed-in example) ──
P3_w = np.linalg.matrix_power(weather_P, 3)
print("P³ =\n", P3_w.round(3))
print(f"P(sunny Sat and Sun | sunny Wed) = (P³)[S,S] · P[S,S] = {P3_w[0, 0]:.3f} · 0.7 = {P3_w[0, 0] * 0.7:.4f}")

rng = np.random.default_rng(SEED + 102)
n_weeks = 100_000
state = np.zeros(n_weeks, dtype=int)                          # Wednesday: sunny (0) in every simulated week
days = []
for _ in range(4):                                            # Thursday, Friday, Saturday, Sunday
    state = np.where(rng.random(n_weeks) < weather_P[state, 0], 0, 1)
    days.append(state)
both = (days[2] == 0) & (days[3] == 0)
print(f"{n_weeks:,} SIMULATED weeks: fraction with sunny Saturday and Sunday = {both.mean():.4f}")

🔍 **What just happened?** $P^3 = \begin{bmatrix}0.667 & 0.333\\ 0.666 & 0.334\end{bmatrix}$, so the probability that it
will be sunny on Saturday is 0.667 and the answer is $0.667 \times 0.7 = 0.4669$. In 100,000 simulated weeks (each
starting with a sunny Wednesday) the fraction with a sunny weekend matches it to about two decimals. Notice that the two
rows of $P^3$ are almost identical: after three days the weather has nearly **forgotten** where it started. That
observation leads to the long-run behaviour of the chain.

### 10.3 The stationary distribution

A probability row vector $\boldsymbol\pi$ is a **stationary distribution** if one more step does not change it:

$$\boldsymbol\pi = \boldsymbol\pi P, \qquad \sum_i \pi_i = 1.$$

*In words:* if the state is distributed as $\boldsymbol\pi$ today, it is distributed as $\boldsymbol\pi$ tomorrow too.
Transposing, $P^\top\boldsymbol\pi^\top = 1\cdot\boldsymbol\pi^\top$: **$\boldsymbol\pi$ is an eigenvector of $P^\top$ (a
"left eigenvector" of $P$) with eigenvalue 1** ([Module 12 · 🌀 Eigenvalues and eigenvectors](12_eigenvalues_eigenvectors.ipynb)), rescaled to sum to 1. For the weather chain by hand:
$\pi_S = 0.7\pi_S + 0.6\pi_C$ gives $0.3\pi_S = 0.6\pi_C$, so $\pi_S = 2\pi_C$, and with $\pi_S + \pi_C = 1$,
$\boldsymbol\pi = (2/3, 1/3)$ — in the long run two days out of three are sunny. For chains like ours (every state can
reach every other, no periodic cycling), $\mathbf v P^n \to \boldsymbol\pi$ from **any** starting distribution, every row
of $P^n$ converges to $\boldsymbol\pi$, and the fraction of time a long simulation spends in each state converges to
$\boldsymbol\pi$ as well.

In [ ]:
# ── 10.3 Stationary distributions: eigenvector, limit of Pⁿ, long simulation — and base composition of DNA ──
def stationary_distribution(P) -> np.ndarray:
    """Left eigenvector of P with eigenvalue 1 (eigenvector of Pᵀ), rescaled to sum to 1."""
    eigenvalues, eigenvectors = np.linalg.eig(P.T)
    k = np.argmin(np.abs(eigenvalues - 1))
    pi = np.real(eigenvectors[:, k])
    return pi / pi.sum()


pi_w, pi_tel = stationary_distribution(weather_P), stationary_distribution(P_tel)
print("weather:   π =", pi_w.round(4), "| π P =", (pi_w @ weather_P).round(4), "| rows of P^20:", np.linalg.matrix_power(weather_P, 20).round(4).tolist())
long_run = simulate_chain(weather_P, [1.0, 0.0], 20_000, np.random.default_rng(SEED + 103))
print(f"           fraction of sunny days in 20,000 SIMULATED days: {np.mean(long_run == 0):.4f}")
print("telephone: π =", pi_tel.round(3), "| max |π P − π| =", f"{np.abs(pi_tel @ P_tel - pi_tel).max():.1e}")
print("           eigenvalue moduli of P:", np.sort(np.abs(np.linalg.eigvals(P_tel))).round(3))
for name, M in [("CpG island", P_c), ("background", P_b)]:
    pi_dna = stationary_distribution(M)
    print(f"{name:10s} chain: long-run base composition A, C, G, T = {pi_dna.round(3)}, GC content {pi_dna[1] + pi_dna[2]:.3f}")

dist = v.copy()
gaps = []
for n in range(1, 16):
    dist = dist @ P_tel
    gaps.append(np.abs(dist - pi_tel).sum())
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogy(range(1, 16), gaps, "o-", color="tab:blue", label="Σ|v Pⁿ − π|")
ax.semilogy(range(1, 16), gaps[0] * 0.35 ** np.arange(15), "--", color="0.5", label="∝ 0.35ⁿ")
ax.set_xlabel("number of steps n"); ax.set_ylabel("distance to π (log scale)")
ax.set_title("telephone chain: v Pⁿ converges to the stationary distribution"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** For the weather chain the eigenvector method gives $\boldsymbol\pi = (0.6667, 0.3333)$, which
satisfies $\boldsymbol\pi P = \boldsymbol\pi$; both rows of $P^{20}$ equal $\boldsymbol\pi$; and a 20,000-day simulation
is sunny on about 67 % of days — three views of the same long-run answer. For the telephone chain the stationary
distribution is about $(0.119, 0.193, 0.186, 0.173, 0.196, 0.133)$: in the long run, all lines are free about 12 % of the
time. The plot shows $\mathbf v P^n$ approaching $\boldsymbol\pi$ geometrically, at the rate of the second-largest
eigenvalue modulus (about 0.35): after 3 steps the distance is already small, which is why the rows of $P^3$ looked so
similar. For DNA, the stationary distribution is the **long-run base composition** a chain produces: the CpG-island
chain generates DNA with a GC content of about 0.69, the background chain about 0.49 — close to the GC contents of the
simulated genome in Section 0.1 and in line with the classic definition of CpG islands (GC content above 50 %, observed/expected CpG
above 0.6, length at least 200 bp; Gardiner-Garden and Frommer, 1987).

**Applications of Markov chains in genomics.** They are widely used to model elements of the genome sequence: finding
**motifs** for transcription-factor binding sites (position-specific models of short words); **hidden Markov models
(HMMs)** for modelling **recombination sites and haplotypes** (the hidden state is the ancestral chromosome a stretch was
copied from) and for modelling **genes with start/stop codons, introns and splice sites** (the hidden state is "exon",
"intron", "intergenic", …).

✏️ **Your turn.** (1) Change the weather matrix to $\begin{bmatrix}0.9 & 0.1\\ 0.5 & 0.5\end{bmatrix}$ and find
$\boldsymbol\pi$ by hand and with `stationary_distribution`. (2) For the telephone chain, what is the probability that no
line is in use at the 10th observation? Compare $\mathbf v P^9$ with $\boldsymbol\pi$.

### 🌉 Bridge — hidden Markov models: gene finders and haplotypes

Section 9 used two separate chains and a sliding window. A **hidden Markov model** (HMM) puts both chains into one
model: a hidden state $H_t \in \{\text{island}, \text{background}\}$ follows its own Markov chain, and each base is drawn
from the chain of the current state. The **forward algorithm** computes $P(\text{sequence})$ by summing over all hidden
paths with exactly the "sum over intermediate states" of Section 10, and the **Viterbi algorithm** finds the most
probable segmentation — island boundaries without a window ([BiologyToML 19 · Hidden Markov models: inference](../../../../ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb)). Gene finders use the same machinery with states
for exons, introns and splice sites, and population geneticists use it to reconstruct haplotypes and recombination
points; [BiologyToML 20 · Hidden Markov models: learning](../../../../ml/tutorial/BiologyToML/notebooks/20_hmm_learning.ipynb) shows how the transition and emission probabilities are learned from data.

### 📘 Lecture notes — n-step transitions and stationary distributions

**$n$-step transitions.** $\Pr(X_{t+n} = j \mid X_t = i) = (P^n)_{ij}$; Chapman–Kolmogorov $P^{m+n} = P^mP^n$; if
$\mathbf v$ is the distribution of $X_1$, then $X_{1+n}$ has distribution $\mathbf vP^n$.

**Telephone lines (states 0–5).** $\mathbf v = (0.5, 0.3, 0.2, 0, 0, 0)$ for $X_1$. All five lines busy now → no line
busy three periods later: $(P^3)_{5,0} = 0.116$. Distribution of $X_3$: $\mathbf vP^2 = (0.133, 0.227, 0.202, 0.156,
0.162, 0.120)$ ($\mathbf vPP = \mathbf vP^2$; two steps after $X_1$).

**Weather.** $P = \begin{bmatrix}0.7 & 0.3\\ 0.6 & 0.4\end{bmatrix}$; sunny Wednesday → $(P^3)_{SS} = 0.667$ for
Saturday; sunny Saturday **and** Sunday: $0.667 \times 0.7 = 0.4669$.

**Stationary distribution.** $\boldsymbol\pi = \boldsymbol\pi P$, $\sum\pi_i = 1$: a left eigenvector of $P$ with
eigenvalue 1, and the limit of the rows of $P^n$. Weather: $\boldsymbol\pi = (2/3, 1/3)$.

**Applications.** Motif finding; HMMs for recombination and haplotypes; HMMs for genes (start/stop codons, introns,
splice sites); CpG islands.

> 🎓 **Exam tip.** Before using $P^n$, write down which time the given distribution refers to and count the steps to the
> time you are asked about. For "both on day $a$ and on day $b$" multiply an $n$-step probability by a one-step one — do
> not square $P^n$.

## 📌 Key takeaways

- A **Bayesian network** is a directed acyclic graph plus one conditional distribution per node:
  $p(\mathbf x) = \prod_k p(x_k \mid \mathrm{Pa}_k)$, parents = in-neighbours. Factorisation → graph: arrows from the
  conditioning variables; graph → factorisation: one factor per node. The graph depends on the chosen factorisation.
- Missing arrows are **conditional-independence assumptions**; they shrink $2^n - 1$ parameters to
  $\sum_k 2^{|\mathrm{Pa}_k|}$ (31 → 10 for the bronchitis and five-node networks).
- Two parents with no common ancestor are **marginally independent** but become **dependent given their child**
  (explaining away).
- Any query can be answered by **enumeration** (sum the joint over matching rows) and checked by **ancestral sampling**;
  $P(T \mid S) = 0.35 \cdot 0.8 + 0.1 \cdot 0.2 = 0.30$.
- Graphical models come as directed networks, **Markov random fields** ($\frac1Z\prod_C\psi_C$) and **factor graphs**.
  The **Markov blanket** (parents, children, co-parents) shields a node from everything else.
- A **Markov chain**: the future is independent of the past given the present; $p(x_1, \dots, x_n) = p(x_1)\prod
  p(x_k \mid x_{k-1})$; a transition matrix with rows summing to 1.
- Two chains score DNA: $P(\text{ACTTC} \mid \text{non-CpG}) / P(\text{ACTTC} \mid \text{CpG}) \approx 1.26$; summed
  log-odds in sliding windows find a CpG island.
- $n$-step probabilities are entries of $P^n$ (telephone: 0.116; weather: $0.667 \times 0.7 = 0.4669$); the
  **stationary distribution** $\boldsymbol\pi = \boldsymbol\pi P$ is a left eigenvector with eigenvalue 1 and the
  long-run behaviour of the chain.

---
## ✅ Checkpoint

1. Draw the Bayesian network of $p(a, b, c, d) = p(a)\,p(b)\,p(c \mid a, b)\,p(d \mid c)$ and count its parameters for binary variables.
   <details><summary>Answer</summary>Arrows a → c, b → c, c → d. Parameters: 1 (a) + 1 (b) + 4 (c | a, b) + 2 (d | c) = 8, versus 2⁴ − 1 = 15 for the full joint.</details>
2. Write the factorisation implied by the arrows $x_5 \to x_2$, $x_1 \to x_3$, $x_2 \to x_3$, $x_2 \to x_4$.
   <details><summary>Answer</summary>p(x₁) p(x₅) p(x₂ | x₅) p(x₃ | x₁, x₂) p(x₄ | x₂) — one factor per node, conditioned on its parents.</details>
3. In the network $X \to Y \to Z$, $X \to Z$ with the CPTs of Section 3, compute $P(X = 1, Y = 1, Z = 1)$.
   <details><summary>Answer</summary>P(X=1) P(Y=1 | X=1) P(Z=1 | X=1, Y=1) = 0.3 × 0.9 × 0.4 = 0.108.</details>
4. Prove that in $B \to A \leftarrow E$ the parents are independent.
   <details><summary>Answer</summary>P(b, e) = Σₐ p(b) p(e) p(a | b, e) = p(b) p(e) Σₐ p(a | b, e) = p(b) p(e), because a conditional distribution sums to 1 over its own variable.</details>
5. In the bronchitis network compute $P(T \mid \neg S)$.
   <details><summary>Answer</summary>P(T | ¬S) = P(T | C, ¬S) P(C) + P(T | ¬C, ¬S) P(¬C) = 0.25 × 0.8 + 0.05 × 0.2 = 0.20 + 0.01 = 0.21 (using C ⊥ S).</details>
6. Using Q5 and $P(T \mid S) = 0.30$, compute $P(T)$.
   <details><summary>Answer</summary>Law of total probability: P(T) = P(T | S) P(S) + P(T | ¬S) P(¬S) = 0.30 × 0.4 + 0.21 × 0.6 = 0.12 + 0.126 = 0.246.</details>
7. Give the Markov blanket of $T$ and of $C$ in the bronchitis network.
   <details><summary>Answer</summary>MB(T) = {S, C} (parents) ∪ {O, A} (children) = {S, C, O, A}; T has no co-parents because O and A have no other parents. MB(C) = {T} (child) ∪ {S} (co-parent) = {S, T}.</details>
8. Under the CpG-island chain with a uniform first letter, compute $P(\text{CGA})$; under the background chain too. Which model is favoured, and by what ratio?
   <details><summary>Answer</summary>CpG: 0.25 × P(G | C) × P(A | G) = 0.25 × 0.27 × 0.16 = 0.0108. Background: 0.25 × 0.08 × 0.25 = 0.005. Ratio CpG/background = 2.16 — the island model is favoured, thanks to the CG pair.</details>
9. For the weather chain, if today is cloudy, what is the probability that tomorrow is sunny and the day after is cloudy?
   <details><summary>Answer</summary>P(cloudy → sunny) × P(sunny → cloudy) = 0.6 × 0.3 = 0.18.</details>
10. Find the stationary distribution of $P = \begin{bmatrix}0.9 & 0.1\\ 0.2 & 0.8\end{bmatrix}$.
    <details><summary>Answer</summary>π₁ = 0.9π₁ + 0.2π₂ ⇒ 0.1π₁ = 0.2π₂ ⇒ π₁ = 2π₂; with π₁ + π₂ = 1, π = (2/3, 1/3).</details>
11. In the telephone example, someone computes the distribution of $X_3$ as $\mathbf vP^3$. Explain the mistake.
    <details><summary>Answer</summary>v is the distribution of X₁, so X₃ is two steps later: vP². vP³ would be the distribution of X₄. (If v were the distribution of X₀, then vP³ would be correct.)</details>
12. Why must a Bayesian network be acyclic, while a transition diagram of a Markov chain (sunny ⇄ cloudy) may have cycles?
    <details><summary>Answer</summary>A Bayesian network's nodes are random variables, and a directed cycle would make the product of conditionals inconsistent (not a valid joint in general). A transition diagram's nodes are *states* of one variable at successive times; unrolled over time, the chain's Bayesian network X₁ → X₂ → … is acyclic.</details>

## 🧠 Questions & discussion

1. A Bayesian network of genes is fitted to expression data and contains the arrow TF → target. What extra assumptions
   would you need before claiming that knocking down TF changes the target? What experiment would test it?
2. The Markov assumption "forgets" everything but the previous base. Which biological features of DNA (codons, motifs,
   nucleosome positioning, long-range enhancer contacts) violate it, and how would a second-order chain (each base
   depending on the previous two) change the number of parameters?
3. Explaining away predicts that, among patients hospitalised with a disease, two independent risk factors appear
   negatively associated. Can you find an example in the literature where such a selection effect caused confusion?
4. The CpG-island chains were estimated from human sequence. Would you use them unchanged for a bacterial genome with
   a GC content of 70 %? What would go wrong?
5. When is it reasonable to treat a cell's progression through states (e.g. the cell cycle, or differentiation) as a
   Markov chain, and what does the stationary distribution mean biologically?

## 🚀 Bonus — going further

- **Jukes–Cantor evolution as a 4-state Markov chain.** Let a site mutate to each of the three other bases with
  probability $\alpha$ per generation: $P = (1 - 3\alpha)I + \alpha(J - I)$, where $J$ is the all-ones matrix. Compute
  $P^n$ for $\alpha = 10^{-3}$ and $n = 1{,}000$, and check that the probability of still having the original base is
  $\tfrac14 + \tfrac34(1 - 4\alpha)^n$. What is the stationary distribution?
- **Variable elimination.** Compute $P(O)$ in the bronchitis network *without* building the 32-row joint: sum out $S$ and
  $C$ first to get $P(T)$, then sum out $T$. Count the multiplications you saved.
- **Learning a chain.** Estimate a transition matrix from `simulate_dna(50_000, CPG_ISLAND_TRANSITIONS)` by counting
  dinucleotides and normalising each row (the maximum-likelihood estimate, [Module 23 · 🎯 Parameter estimation and maximum likelihood](23_maximum_likelihood.ipynb)). How close are you to the truth?
- **Further reading.** DeGroot & Schervish, *Probability and Statistics*, §3.10 (Markov chains); Bishop, *Pattern
  Recognition and Machine Learning*, Chapter 8 (graphical models); Durbin *et al.*, *Biological Sequence Analysis*,
  Chapter 3 (Markov chains, CpG islands and HMMs).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 20 — Graphical models: Bayesian networks and Markov chains — date:
Input: factorisations and graphs (a,b,c; five-node; X/Y/Z); CPTs of the alarm and bronchitis networks;
       the Markov-blanket network; the CpG/background transition matrices; a SIMULATED genome with one island;
       the telephone-line and weather chains
Output: graphs ⇄ factorisations, parameter counts, joint tables, P(T|S) = 0.30 and other queries,
        Markov blankets, ACTTC likelihood ratio 1.26, window scores, P², P³, stationary distributions
Method: p(x) = Π p(x_k | Pa_k); enumeration; ancestral sampling; Markov factorisation; log-odds; matrix powers; eigenvectors
Key equation: p(x) = Π_k p(x_k | Pa_k);  p(x_1..x_n) = p(x_1) Π p(x_k | x_(k-1));  π = π P
Key parameters: CPT entries; transition matrices; window length; number of steps n
What changed when I modified the CPTs / the window length / the transition matrix:
One failure mode: (e.g., forgetting co-parents in a Markov blanket; reading P by columns; off-by-one steps in P^n)
Connection to 02-680 (which part of the topic on graphical representations of random variables this illustrates):
```

## 📚 References

- Bishop C. M. (2006) *Pattern Recognition and Machine Learning*. Springer — Chapter 8 (Bayesian networks, the
  five-node example, Markov random fields, factor graphs, Markov blankets).
- Deisenroth M. P., Faisal A. A., Ong C. S. (2020) *Mathematics for Machine Learning*. Cambridge University Press —
  §8.5 (directed graphical models: constructing networks from factorisations and reading factorisations off graphs).
- DeGroot M. H., Schervish M. J. (2012) *Probability and Statistics*, 4th ed. Pearson — §3.10 (Markov chains: the
  telephone-lines and weather examples, transition matrices, stationary distributions).
- Wasserman L. (2004) *All of Statistics*. Springer — Chapters 17 (directed graphs) and 18 (undirected graphs), §23.2
  (Markov chains).
- Pearl J. (2009) *Causality: Models, Reasoning, and Inference*, 2nd ed. Cambridge University Press.
- Durbin R., Eddy S., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis*. Cambridge University Press —
  Chapter 3 (Markov chains, CpG islands, hidden Markov models).
- Gardiner-Garden M., Frommer M. (1987) CpG islands in vertebrate genomes. *Journal of Molecular Biology* 196:261–282.
- Sachs K., Perez O., Pe'er D., Lauffenburger D. A., Nolan G. P. (2005) Causal protein-signaling networks derived from
  multiparameter single-cell data. *Science* 308:523–529.
- D. DeBlasio, 02-680 lecture notes, github.com/deblasiolab/02-680_Documents.

## ➡️ Next

[Module 21 · ⚖️ Expectation, variance and moments](21_expectation_variance_moments.ipynb) — so far we have described random variables by whole distributions: tables, densities,
networks and chains. The next module summarises a distribution by a few numbers — its **expectation** (the long-run
average, the centre of gravity), its **variance** and higher **moments** — proves the rules for sums and linear
transformations that Section 10's long-run averages and the covariances of [Module 19 · 🔗 Several random variables: joint, marginal, conditional](19_multiple_random_variables.ipynb) relied on, and introduces
**conditional expectation**, the law of total expectation and the law of total variance, with highly variable genes in
single-cell data as the biological payoff.